# Standardized SME forecasting-to-inventory evaluation pipeline

This notebook implements a standardized forecasting-to-decision pipeline for SME demand forecasting and inventory evaluation. The analytical sequence and model logic are held constant across applications; the Shopify export and the case-specific operating assumptions in `CONFIG` are the inputs that change between businesses.

Set `CONFIG["csv_path"]` to the location of a Shopify order-export CSV. The file can be stored anywhere on the user's computer; no specific project-folder structure is required. If `csv_path` is left as `None`, the pipeline falls back to `data/raw/<raw_file_name>` for compatibility with the original project structure.

The pipeline proceeds from raw transaction data to weekly SKU demand, portfolio characterization and leakage-safe sample construction, rolling out-of-sample forecasts, protection-horizon evaluation, method-specific safety-stock calibration, and a common periodic-review inventory simulation. It then tests the robustness of the operational result across service targets and operating conditions.

The forecasting comparison includes:
- lower-complexity methods: naive, seasonal naive, moving average, ETS, and TSB;
- standardized ML: gradient boosting and Random Forest trained one week ahead and recursively extended;
- decision-aligned ML: direct-H gradient boosting and Random Forest trained on cumulative demand over the case-specific protection horizon `H = lead time + review period`.

All forecasting methods are translated through the same inventory-policy logic. The final evaluation window is defined in `CONFIG`, while tuning, sample construction, revenue weighting, and safety-stock calibration use only information available before that window.


## 1. Setup

Initializes runtime tracking, reproducibility checks, and conservative numerical-library thread limits. The helper functions defined here collect warnings and failures so that the final health check can summarize whether a run is suitable for interpretation. No data are transformed in this section.


In [ ]:
from pathlib import Path
import os
import re
import time
import warnings

NOTEBOOK_START_TIME = time.time()

# Keep numerical libraries controlled so the notebook runs reliably on normal laptops.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")


## 2. Case-specific configuration

This is the only section that should normally be edited from one SME case to another. It defines the Shopify input file, regional CSV conventions, final out-of-sample evaluation length, minimum training history, operating assumptions, sensitivity grids, and computational sampling limits.

Lead time, review frequency, service target, and order constraints should be set from case evidence where available. `holding_cost` is a normalized reference scale and `stockout_cost_mult` expresses shortage cost relative to that scale; they are used for controlled method comparison rather than as a full accounting estimate. A positive `order_batch_multiplier` represents a batch size equal to that many weeks of the SKU's pre-evaluation average demand; `0` disables batch rounding.


In [ ]:
CONFIG = {
    # ------------------------------------------------------------------
    # INPUT DATA
    # ------------------------------------------------------------------
    # Easiest option: set the path to the Shopify order-export CSV.
    # Examples:
    #   "shopify_orders_export.csv"
    #   "/Users/yourname/Downloads/orders_export.csv"
    #   r"C:\\Users\\yourname\\Downloads\\orders_export.csv"
    # Leave as None to use the default project structure: data/raw/<raw_file_name>.
    "csv_path": None,

    # Used only when csv_path = None.
    "raw_file_name": "shopify_orders_export.csv",

    # Regional CSV format.
    "italian_format": True,       # True = prefer ';', DD/MM/YYYY, comma decimals and dot thousands
    "csv_separator": "auto",     # "auto", ";", or ","
    "csv_encoding": "utf-8-sig", # Handles standard UTF-8 and Excel/Shopify BOM files

    # Chronological evaluation settings.
    "evaluation_weeks": 12,       # Final out-of-sample window used for both forecast accuracy and inventory simulation
    "min_train_weeks": 26,        # Required strictly before the final evaluation window

    # Operational reference scenario.
    "lead_time_weeks": 2,          # FROM INTERVIEW: "Tempo di approvvigionamento"
    "review_period_weeks": 2,      # FROM INTERVIEW: "Frequenza di riordino"
    "holding_cost": 1.0,           # Reference scale
    "stockout_cost_mult": 0.5,     # Explored across the sensitivity grid below
    "service_level": 0.95,         # FROM INTERVIEW: "Livello di servizio desiderato"
    "safety_stock_method": "empirical",  # "empirical" main specification; "normal" robustness
    "order_batch_multiplier": 0.0, # 0 = no MOQ / batch rounding
    "terminal_holding_weeks": 1.0,

    # Scenario grids.
    "lead_grid": [1, 2, 4, 8],
    "stockout_grid": [0.5, 1, 10, 20, 40],
    "batch_grid": [0.0, 1.0, 3.0],
    "service_grid": [0.80, 0.85, 0.90, 0.95],

    # Controlled comparison requirement.
    "min_common_revenue_coverage": 0.80,

    # ABC-XYZ thresholds and computational sampling.
    "abc_cum_thresholds": (0.80, 0.95),
    "xyz_cv_thresholds": (0.5, 1.0),
    "max_skus_total_a": 6600,
    "max_skus_per_bc_cell": 400,
    "sampling_random_seed": 42,

    # Runtime controls.
    "refit_every_n_origins": 1,
    "ml_n_jobs": 4,
}

MAX_PROTECTION_HORIZON = max(CONFIG["lead_grid"]) + CONFIG["review_period_weeks"]
print("Maximum protection horizon used in the notebook:", MAX_PROTECTION_HORIZON, "weeks")
print("Input format:", "Italian" if CONFIG["italian_format"] else "International / Shopify standard")
print(
    "\nReminder before treating results as calibrated to this SME: confirm "
    "holding_cost, stockout_cost_mult, service_level, lead_time_weeks, "
    "review_period_weeks, and order_batch_multiplier against the interview."
)


## 3. Libraries and run folders

Loads the analytical libraries and creates versioned folders for tables, figures, and section logs. Output paths are derived from the input filename so the same notebook can be run independently for different SME cases.


In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 180)
warnings.filterwarnings("ignore")

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = PROJECT_DIR / "data" / "raw"
DATA_PROCESSED = PROJECT_DIR / "data" / "processed"
OUTPUTS_ROOT = PROJECT_DIR / "outputs"

# Input CSV:
# - if csv_path is provided, use it directly;
# - otherwise fall back to the original data/raw project structure.
if CONFIG.get("csv_path"):
    RAW_FILE = Path(CONFIG["csv_path"]).expanduser()
    if not RAW_FILE.is_absolute():
        RAW_FILE = (Path.cwd() / RAW_FILE).resolve()
    else:
        RAW_FILE = RAW_FILE.resolve()
else:
    RAW_FILE = DATA_RAW / CONFIG["raw_file_name"]

PIPELINE_VERSION = "v1.0.0"
CASE_ID = re.sub(r"[^A-Za-z0-9_-]+", "_", RAW_FILE.stem).strip("_") or "case"
RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
RUN_ID = os.environ.get("PIPELINE_RUN_ID", f"{CASE_ID}_{PIPELINE_VERSION}_{RUN_TIMESTAMP}")

# Every execution writes to a fresh folder so outputs from older notebook
# versions cannot be mistaken for the current results. No CSV tables are
# saved by this pipeline version -- only a plain-text log per section
# (below) and the figures saved directly by the plotting cells.
OUTPUTS = OUTPUTS_ROOT / "runs" / RUN_ID
FIGURES = OUTPUTS / "figures"
LOGS = OUTPUTS / "logs"
for d in (DATA_RAW, DATA_PROCESSED, FIGURES, LOGS):
    d.mkdir(parents=True, exist_ok=True)

import re as _re
import sys


class _SectionLogger:
    """Splits printed output into one text file per notebook section.

    Everything still prints in the notebook exactly as before. In addition,
    calling section(number, title) closes whichever section file was open
    and starts a new one at logs/<number>_<title>.txt, so every block's
    printed output (SKU counts, scoreboards, warnings, the final health
    check, etc.) ends up saved as its own readable text file -- with no CSVs
    and nothing that needs a separate manual save step.
    """

    def __init__(self, base_stdout):
        self.base_stdout = base_stdout
        self.current_file = None

    def start(self, number, title):
        if self.current_file is not None:
            self.current_file.flush()
            self.current_file.close()
        slug = _re.sub(r"[^a-z0-9]+", "_", title.lower()).strip("_") or "section"
        path = LOGS / f"{number:02d}_{slug}.txt"
        self.current_file = open(path, "w", encoding="utf-8")
        header = f"=== Section {number}: {title} ===\n"
        self.base_stdout.write(header)
        self.current_file.write(header)

    def write(self, data):
        self.base_stdout.write(data)
        if self.current_file is not None:
            self.current_file.write(data)

    def flush(self):
        self.base_stdout.flush()
        if self.current_file is not None:
            self.current_file.flush()

    def close(self):
        if self.current_file is not None:
            self.current_file.flush()
            self.current_file.close()
            self.current_file = None


_section_logger = _SectionLogger(sys.stdout)
sys.stdout = _section_logger
sys.stderr = _section_logger


def section(number, title):
    _section_logger.start(number, title)


RUN_DIAGNOSTICS = []

def check(name, ok, message, level="WARN"):
    tag = "PASS" if ok else level
    RUN_DIAGNOSTICS.append({"check": name, "status": tag, "message": message})
    print(f"[{tag}] {name}: {message}")


section(3, "Libraries and run folders")
print(f"Per-section text logs will be saved under: {LOGS}")
print("Project folder:", PROJECT_DIR)
print("Raw file:", RAW_FILE)
print("Raw file exists:", RAW_FILE.exists())
print("Pipeline version:", PIPELINE_VERSION)
print("Run ID:", RUN_ID)
print("Run-specific output folder:", OUTPUTS)


## 4. Load the raw Shopify orders export

Reads the line-level Shopify order export from the path defined by `CONFIG["csv_path"]`, or from `data/raw/<raw_file_name>` when `csv_path` is left as `None`. The separator, encoding, and locale settings are also defined in `CONFIG`. The parser checks that the file is available and loads the raw fields before any filtering or aggregation is performed.


In [ ]:
section(4, "Load the raw Shopify orders export")
if not RAW_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find the Shopify CSV at:\n{RAW_FILE}\n\n"
        "Set CONFIG['csv_path'] to the location of your Shopify order-export CSV, "
        "or place the file in data/raw/ and set CONFIG['raw_file_name']."
    )


def read_shopify_csv(path):
    """Read only the Shopify columns actually required by the pipeline."""

    configured_sep = CONFIG.get("csv_separator", "auto")
    encoding = CONFIG.get("csv_encoding", "utf-8-sig")

    # Only these columns are used later by clean_orders()
    wanted_cols = {
        "name",
        "created at",
        "financial status",
        "cancelled at",
        "lineitem sku",
        "lineitem name",
        "lineitem quantity",
        "lineitem price",
        "lineitem discount",
    }

    # Case-insensitive column selection
    def keep_column(col):
        return col.strip().lower() in wanted_cols

    def _read(sep, engine=None):
        kwargs = {
            "dtype": str,
            "sep": sep,
            "encoding": encoding,
            "usecols": keep_column,
        }
        if engine is not None:
            kwargs["engine"] = engine

        return pd.read_csv(path, **kwargs)

    if configured_sep != "auto":
        df = _read(configured_sep)

    else:
        preferred_sep = ";" if CONFIG.get("italian_format", False) else ","

        df = _read(preferred_sep)

        if df.shape[1] <= 1:
            df = _read(None, engine="python")

        if df.shape[1] <= 1:
            df = _read(";" if preferred_sep != ";" else ",")

    return df


raw = read_shopify_csv(RAW_FILE)
print("Rows:", len(raw), "| Columns:", len(raw.columns))
print("Detected columns:", list(raw.columns))
check(
    "Raw file loaded",
    len(raw) > 0 and len(raw.columns) > 5,
    f"{len(raw)} rows and {len(raw.columns)} columns detected",
    level="FAIL",
)
raw.head(8)


## 5. Clean raw orders into fulfilled sales lines

Converts the raw Shopify export into a consistent sales-line table. The routine forward-fills order-level fields where required, excludes cancelled or non-valid transactions, parses dates and numeric values using the configured locale, removes invalid quantities/SKUs, and constructs gross value, discount amount, and promotion indicators.

The output represents observed sales. Historical lost demand is not imputed when stock was unavailable.


In [ ]:
section(5, "Clean raw orders into fulfilled sales lines")
def require_columns(df, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected Shopify columns: {missing}")


def normalize_shopify_columns(raw, required):
    """Match required Shopify columns case-insensitively and rename to canonical form.

    Real Shopify exports are not always consistent in header casing across
    stores/locales (e.g. "Lineitem sku" vs "Lineitem SKU"). Find each
    required column case-insensitively and rename it to the canonical name
    the rest of the pipeline expects, instead of crashing on a casing
    difference. Only genuinely missing columns raise an error, and the
    error lists the columns actually present in the file.
    """
    lower_lookup = {c.lower(): c for c in raw.columns}
    rename_map, missing = {}, []
    for req in required:
        actual = lower_lookup.get(req.lower())
        if actual is None:
            missing.append(req)
        elif actual != req:
            rename_map[actual] = req
    if missing:
        raise ValueError(
            f"Missing expected Shopify columns: {missing}. "
            f"Columns found in file: {list(raw.columns)}"
        )
    return raw.rename(columns=rename_map) if rename_map else raw


def _to_numeric_locale_safe(series):
    """Parse numeric strings that may use comma as the decimal separator.

    Italian-locale exports often write "12,50" instead of "12.50", and
    sometimes "1.234,56" with a dot as the thousands separator. Passed
    straight to pd.to_numeric, either of those does not raise an error --
    it silently becomes NaN, which then quietly corrupts revenue, discount
    amounts, and on_promo for however many rows are affected. Detect values
    that look like a comma-decimal number and normalize them before
    conversion, so a locale difference doesn't turn into silent missing
    data. Handles a bare scalar (e.g. the default 0 for a missing column)
    the same way pd.to_numeric always did.
    """
    if not isinstance(series, pd.Series):
        return pd.to_numeric(series, errors="coerce")
    s = series.astype(str).str.strip()
    looks_comma_decimal = s.str.match(r"^-?\d{1,3}(\.\d{3})*,\d+$") | s.str.match(r"^-?\d+,\d+$")
    normalized = s.mask(
        looks_comma_decimal,
        s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
    )
    return pd.to_numeric(normalized, errors="coerce")


def clean_orders(raw):
    required = [
        "Name", "Created at",
        "Lineitem SKU", "Lineitem name", "Lineitem quantity", "Lineitem price", "Lineitem discount",
    ]
    df = normalize_shopify_columns(raw, required).copy()

    # Normalize optional status columns case-insensitively as well.
    optional_status_cols = ["Financial Status", "Cancelled at"]
    lower_lookup = {c.lower(): c for c in df.columns}

    optional_rename = {}
    for canonical in optional_status_cols:
        actual = lower_lookup.get(canonical.lower())
        if actual is not None and actual != canonical:
            optional_rename[actual] = canonical

    if optional_rename:
        df = df.rename(columns=optional_rename)
        
    # Shopify often stores order-level fields only on the first row of a multi-line order.
    df["Name"] = df["Name"].replace("", pd.NA).ffill()

    # Created at is always required.
    df["Created at"] = df["Created at"].replace("", pd.NA)
    df["Created at"] = df.groupby("Name", sort=False)["Created at"].ffill()


    # ------------------------------------------------------------
    # Optional order-status filtering
    # ------------------------------------------------------------

    # Financial Status and Cancelled at are treated as an all-or-nothing pair.
    # If BOTH are available, apply the original Shopify order-validity filters.
    # If either is missing, skip BOTH filters and continue.
    status_cols_available = (
        "Financial Status" in df.columns
        and "Cancelled at" in df.columns
    )

    if status_cols_available:

        # Forward-fill order-level status fields across multi-line orders.
        for col in ["Financial Status", "Cancelled at"]:
            df[col] = df[col].replace("", pd.NA)
            df[col] = df.groupby("Name", sort=False)[col].ffill()

        # Remove cancelled orders.
        cancelled = (
            df["Cancelled at"].notna()
            & (df["Cancelled at"].astype(str).str.strip() != "")
        )
        df = df[~cancelled].copy()

        # Retain paid / partially refunded orders.
        status = (
            df["Financial Status"]
            .astype(str)
            .str.strip()
            .str.lower()
        )
        df = df[
            status.isin(["paid", "partially_refunded"])
        ].copy()

        print(
            "Order-status filters applied "
            "(Financial Status + Cancelled at available)."
        )

    else:
        print(
            "Warning: Financial Status and/or Cancelled at missing. "
            "Both payment-status and cancellation filters were skipped."
        )

    # dayfirst=True: Italian-locale exports write dates as DD/MM/YYYY. Without this,
    # any date where the day is <=12 (so it also looks like a valid month) gets
    # silently parsed as MM/DD instead of erroring, shifting that order into the
    # wrong week for the rest of the pipeline.
    clean = pd.DataFrame({
        "date": pd.to_datetime(
            df["Created at"],
            errors="coerce",
            utc=True,
        ).dt.tz_convert(None),
        "sku": df["Lineitem SKU"].astype(str).str.strip(),
        "name": df["Lineitem name"].astype(str).str.strip(),
        "quantity": _to_numeric_locale_safe(df["Lineitem quantity"]),
        "price": _to_numeric_locale_safe(df["Lineitem price"]),
    })

    # .astype(str) on a genuinely missing value produces the literal string
    # "nan" (a well-known pandas/numpy quirk), which then slips past the
    # `!= ""` filter below and gets treated as a real product code -- silently
    # lumping every row with a truly missing SKU into one fake "nan" product.
    clean["sku"] = clean["sku"].replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})

    n_unparsed_dates = int(clean["date"].isna().sum())
    if n_unparsed_dates > 0:
        check(
            "Date parsing",
            n_unparsed_dates / max(len(clean), 1) < 0.05,
            f"{n_unparsed_dates} of {len(clean)} rows could not be parsed and will be dropped. "
            f"Italian format is set to {CONFIG.get('italian_format', False)}.",
        )

    line_disc = _to_numeric_locale_safe(df.get("Lineitem discount", 0)).fillna(0.0)
    clean["gross_value"] = (clean["price"] * clean["quantity"]).fillna(0.0)
    clean["discount_amount"] = (-line_disc).clip(lower=0.0)
    clean["discount_pct"] = np.where(
        clean["gross_value"] > 0,
        clean["discount_amount"] / clean["gross_value"],
        0.0,
    )
    clean["discount_pct"] = clean["discount_pct"].clip(0, 1).round(3)
    clean["on_promo"] = clean["discount_pct"] > 0

    clean = clean[clean["sku"].notna() & (clean["sku"] != "")].copy()
    clean = clean[clean["date"].notna()].copy()
    clean = clean[clean["quantity"] > 0].copy()
    return clean.reset_index(drop=True)


sales = clean_orders(raw)
print("Raw line items:", len(raw))
print("Clean sales rows:", len(sales))
print("Products:", sales["sku"].nunique())
print("Date range:", sales["date"].min().date(), "to", sales["date"].max().date())
print("Total units sold:", int(sales["quantity"].sum()))
print("Promo share:", f"{sales['on_promo'].mean():.1%}")
sales.head()

retention_rate = len(sales) / max(len(raw), 1)
check(
    "Sales cleaning retention",
    retention_rate > 0.30,
    f"{retention_rate:.1%} of raw line items retained after cleaning ({len(sales)} of {len(raw)}).",
)


## 6. Build the weekly SKU demand panel

Aggregates cleaned sales to SKU-week level. For each SKU, the series starts at its first observed sale and subsequent weeks with no recorded sales are entered as zero through the end of the dataset. Weekly units, gross value, discount amount, average discount, and promotion status are retained.

A rule-based `probable_stockout` flag identifies suspicious zero-sales weeks after recent positive demand. The flag is used as contextual evidence only; recorded sales are not replaced with estimated demand.


In [ ]:
section(6, "Build the weekly SKU demand panel")
def build_weekly_panel(sales):
    s = sales.copy()

    s["date"] = pd.to_datetime(
        s["date"],
        errors="coerce",
        utc=True,
    ).dt.tz_convert(None)

    s = s[s["date"].notna()].copy()

    s["week"] = s["date"].dt.to_period("W").dt.start_time

    weekly = (
        s.groupby(["sku", "week"], as_index=False)
         .agg(
             units=("quantity", "sum"),
             gross_value=("gross_value", "sum"),
             discount_amount=("discount_amount", "sum"),
         )
    )
    weekly["avg_discount"] = np.where(
        weekly["gross_value"] > 0,
        weekly["discount_amount"] / weekly["gross_value"],
        0.0,
    ).clip(0, 1)
    weekly["on_promo"] = weekly["avg_discount"] > 0

    global_last_week = weekly["week"].max()
    parts = []
    for sku, g in weekly.groupby("sku", sort=False):
        first_week = g["week"].min()
        sku_weeks = pd.date_range(first_week, global_last_week, freq="W-MON")
        idx = pd.MultiIndex.from_product([[sku], sku_weeks], names=["sku", "week"])
        sku_panel = g.set_index(["sku", "week"]).reindex(idx).reset_index()
        parts.append(sku_panel)

    panel = pd.concat(parts, ignore_index=True)
    panel["units"] = panel["units"].fillna(0).astype(float)
    panel["gross_value"] = panel["gross_value"].fillna(0.0)
    panel["discount_amount"] = panel["discount_amount"].fillna(0.0)
    panel["avg_discount"] = panel["avg_discount"].fillna(0.0).clip(0, 1)
    panel["on_promo"] = panel["on_promo"].fillna(False).astype(bool)
    return panel.sort_values(["sku", "week"]).reset_index(drop=True)


def add_probable_stockout_flags(panel, lookback=4, min_recent_avg=1.0, min_recent_nonzero_weeks=2):
    df = panel.sort_values(["sku", "week"]).copy()
    g = df.groupby("sku")["units"]
    df["recent_avg_units"] = g.transform(lambda x: x.shift(1).rolling(lookback, min_periods=1).mean())
    df["recent_nonzero_weeks"] = g.transform(lambda x: (x.shift(1) > 0).rolling(lookback, min_periods=1).sum())
    df["probable_stockout"] = (
        (df["units"] == 0)
        & (df["recent_avg_units"] >= min_recent_avg)
        & (df["recent_nonzero_weeks"] >= min_recent_nonzero_weeks)
    )
    return df


panel = add_probable_stockout_flags(build_weekly_panel(sales))
print("Panel rows:", len(panel))
print("Products:", panel["sku"].nunique(), "| Calendar weeks:", panel["week"].nunique())
print("Probable stockout flags:", int(panel["probable_stockout"].sum()))
panel.head(10)


check(
    "Weekly demand panel",
    len(panel) > 0 and panel["sku"].nunique() > 0 and panel["week"].nunique() > 0,
    f"{len(panel)} rows, {panel['sku'].nunique()} SKUs, {panel['week'].nunique()} calendar weeks",
    level="FAIL",
)


## 7. Portfolio structure: ABC-XYZ

Builds the portfolio profile used for case interpretation. ABC classes are based on cumulative discount-adjusted revenue contribution, while XYZ classes are based on the coefficient of variation of weekly recorded demand. The full-history matrix is descriptive.

A second profile is constructed using only information available before the final evaluation window. That pre-evaluation profile defines the eligible universe for sampling, preventing the final holdout from influencing which SKUs enter the computational sample.


In [ ]:
section(7, "Portfolio structure: ABC-XYZ")
# Full-history ABC-XYZ is descriptive. Eligibility is determined only with pre-evaluation information.

abc_cum_thresholds = CONFIG.get("abc_cum_thresholds", (0.80, 0.95))
xyz_cv_thresholds = CONFIG.get("xyz_cv_thresholds", (0.5, 1.0))
# Final out-of-sample evaluation window.
# The configured final evaluation window is used for forecast evaluation and inventory simulation.
# Earlier observations are used only for model training and leakage-safe
# safety-stock calibration.
EVALUATION_WEEKS_N = int(CONFIG["evaluation_weeks"])
MIN_TRAIN_WEEKS = int(CONFIG["min_train_weeks"])

ALL_WEEKS = np.sort(panel["week"].unique())
required_calendar_weeks = MIN_TRAIN_WEEKS + EVALUATION_WEEKS_N
if len(ALL_WEEKS) < required_calendar_weeks:
    raise ValueError(
        f"Not enough calendar history for leakage-free sampling and evaluation: "
        f"{len(ALL_WEEKS)} weeks available, need at least {required_calendar_weeks} "
        f"(min_train_weeks + evaluation_weeks)."
    )

EVALUATION_WEEKS = list(ALL_WEEKS[-EVALUATION_WEEKS_N:])
EVALUATION_START = pd.Timestamp(min(EVALUATION_WEEKS))


def classify_abc_xyz(full_panel, abc_cum_thresholds=(0.80, 0.95), xyz_cv_thresholds=(0.5, 1.0)):
    """Classify SKUs by cumulative discount-adjusted line revenue and demand CV."""
    profile = full_panel.groupby("sku").agg(
        gross_revenue=("gross_value", "sum"),
        discount_amount=("discount_amount", "sum"),
        weeks_observed=("week", "nunique"),
        mean_weekly_units=("units", "mean"),
        std_weekly_units=("units", lambda x: np.std(x)),
    ).reset_index()

    profile["discount_adjusted_revenue"] = (
        profile["gross_revenue"] - profile["discount_amount"]
    ).clip(lower=0.0)
    profile["cv"] = np.where(
        profile["mean_weekly_units"] > 0,
        profile["std_weekly_units"] / profile["mean_weekly_units"],
        np.nan,
    )

    profile = profile.sort_values("discount_adjusted_revenue", ascending=False).reset_index(drop=True)
    total_revenue = float(profile["discount_adjusted_revenue"].sum())
    if total_revenue > 0:
        profile["revenue_share"] = profile["discount_adjusted_revenue"] / total_revenue
    else:
        profile["revenue_share"] = 1.0 / len(profile) if len(profile) else np.nan
    profile["cum_revenue_share"] = profile["revenue_share"].cumsum()

    a_cut, b_cut = abc_cum_thresholds
    profile["abc_class"] = np.select(
        [profile["cum_revenue_share"] <= a_cut, profile["cum_revenue_share"] <= b_cut],
        ["A", "B"],
        default="C",
    )

    x_cut, y_cut = xyz_cv_thresholds
    profile["xyz_class"] = np.select(
        [profile["cv"] < x_cut, profile["cv"] <= y_cut],
        ["X", "Y"],
        default="Z",
    )
    profile.loc[profile["cv"].isna(), "xyz_class"] = "Z"
    profile["abc_xyz_cell"] = profile["abc_class"] + profile["xyz_class"]
    return profile.sort_values(
        ["abc_class", "xyz_class", "discount_adjusted_revenue"],
        ascending=[True, True, False],
    ).reset_index(drop=True)


# Full-history profile retained for descriptive case interpretation.
abc_xyz_profile = classify_abc_xyz(panel, abc_cum_thresholds, xyz_cv_thresholds)

sku_count_matrix = (
    abc_xyz_profile.pivot_table(index="abc_class", columns="xyz_class", values="sku", aggfunc="count", fill_value=0)
    .reindex(index=["A", "B", "C"], columns=["X", "Y", "Z"], fill_value=0)
)
revenue_share_matrix = (
    abc_xyz_profile.pivot_table(index="abc_class", columns="xyz_class", values="revenue_share", aggfunc="sum", fill_value=0)
    .reindex(index=["A", "B", "C"], columns=["X", "Y", "Z"], fill_value=0)
)
cell_summary = (
    abc_xyz_profile.groupby(["abc_class", "xyz_class"])
    .agg(n_skus=("sku", "count"), revenue_share=("revenue_share", "sum"))
    .reset_index()
)
cell_summary["sku_share"] = cell_summary["n_skus"] / len(abc_xyz_profile)

# Establish the eligible sampling universe using only information available
# before the final evaluation starts.
pre_evaluation_panel = panel[panel["week"] < EVALUATION_START].copy()
evaluation_panel_all = panel[panel["week"] >= EVALUATION_START].copy()

pre_evaluation_len = pre_evaluation_panel.groupby("sku")["week"].nunique()
evaluation_len_all = evaluation_panel_all.groupby("sku")["week"].nunique()

eligible_pre_evaluation_skus = sorted([
    sku for sku in panel["sku"].unique()
    if pre_evaluation_len.get(sku, 0) >= MIN_TRAIN_WEEKS
    and evaluation_len_all.get(sku, 0) == EVALUATION_WEEKS_N
])

sampling_abc_xyz_profile = classify_abc_xyz(
    pre_evaluation_panel[pre_evaluation_panel["sku"].isin(eligible_pre_evaluation_skus)].copy(),
    abc_cum_thresholds,
    xyz_cv_thresholds,
)
sampling_abc_xyz_profile["sampling_information_cutoff"] = str(EVALUATION_START.date())

print("Full-history SKUs classified for case interpretation:", len(abc_xyz_profile))
print("Pre-evaluation eligible SKU universe:", len(sampling_abc_xyz_profile))
print(
    "Final evaluation window:",
    EVALUATION_START.date(),
    "to",
    pd.Timestamp(EVALUATION_WEEKS[-1]).date(),
)
print("Final evaluation length:", EVALUATION_WEEKS_N, "weeks")

print("\nFull-history SKU count matrix (rows = ABC, columns = XYZ):")
print(sku_count_matrix.to_string())
print("\nFull-history revenue share matrix (rows = ABC, columns = XYZ):")
print(revenue_share_matrix.round(3).to_string())


## 8. Leakage-safe ABC-XYZ SKU sampling

Fixes the computational SKU sample before forecasting. A-tier products are prioritized by pre-evaluation discount-adjusted revenue; B/C cells are retained in full up to the configured cap and otherwise sampled reproducibly within each ABC-XYZ cell. The notebook reports retained SKU and revenue coverage so the scope of the final comparison remains explicit.


In [ ]:
section(8, "Leakage-safe ABC-XYZ SKU sampling")

# Apply the same computational-sampling logic across cases, anchored to
# the PRE-EVALUATION cutoff used by the configured final evaluation design.
# No information from the final evaluation window can influence SKU inclusion.

MAX_SKUS_TOTAL_A = int(CONFIG["max_skus_total_a"])
MAX_SKUS_PER_BC_CELL = int(CONFIG["max_skus_per_bc_cell"])
SAMPLING_RANDOM_SEED = int(CONFIG["sampling_random_seed"])

rng = np.random.default_rng(SAMPLING_RANDOM_SEED)
sampled_parts = []

# A-tier: retain all A SKUs unless the configured portfolio cap is exceeded;
# if it is exceeded, retain the highest pre-evaluation revenue A SKUs.
a_mask = sampling_abc_xyz_profile["abc_xyz_cell"].str.startswith("A")
a_pool = sampling_abc_xyz_profile[a_mask]

if len(a_pool) <= MAX_SKUS_TOTAL_A:
    sampled_parts.append(a_pool)
else:
    sampled_parts.append(
        a_pool.nlargest(MAX_SKUS_TOTAL_A, "discount_adjusted_revenue")
    )

# B/C tiers: retain each ABC-XYZ cell in full up to the configured cap;
# above the cap, draw a reproducible random sample within that cell.
for cell, g in sampling_abc_xyz_profile[~a_mask].groupby("abc_xyz_cell"):
    if len(g) <= MAX_SKUS_PER_BC_CELL:
        sampled_parts.append(g)
    else:
        idx = rng.choice(
            g.index,
            size=MAX_SKUS_PER_BC_CELL,
            replace=False,
        )
        sampled_parts.append(g.loc[idx])

sku_sample = (
    pd.concat(sampled_parts).reset_index(drop=True)
    if sampled_parts
    else sampling_abc_xyz_profile.iloc[0:0].copy()
)

sampled_skus = set(sku_sample["sku"])

# Make retained coverage explicit.
profile_flagged = sampling_abc_xyz_profile.copy()
profile_flagged["sampled"] = profile_flagged["sku"].isin(sampled_skus)

sample_summary = (
    profile_flagged
    .groupby("abc_xyz_cell", group_keys=False)
    .apply(
        lambda g: pd.Series({
            "n_skus_total": len(g),
            "n_skus_sampled": int(g["sampled"].sum()),
            "revenue_share_total": g["revenue_share"].sum(),
            "revenue_share_sampled": g.loc[g["sampled"], "revenue_share"].sum(),
        })
    )
)

retained_sampling_revenue_share = float(
    profile_flagged.loc[profile_flagged["sampled"], "revenue_share"].sum()
)

print("Leakage-safe SKU sampling summary (pre-evaluation ABC-XYZ profile):")
print(sample_summary.round(3).to_string())
print(f"\nSKUs in full descriptive portfolio: {len(abc_xyz_profile)}")
print(f"SKUs in pre-evaluation eligible sampling universe: {len(sampling_abc_xyz_profile)}")
print(f"SKUs sampled for forecasting/simulation: {len(sampled_skus)}")
print(
    "Pre-evaluation revenue share retained within the eligible sampling universe: "
    f"{retained_sampling_revenue_share:.1%}"
)

# All forecasting and simulation sections operate only on the locked sample.
panel = panel[panel["sku"].isin(sampled_skus)].copy()


## 9. Chronological split and usable SKUs

Defines the final chronological evaluation window and retains only sampled SKUs with the required pre-evaluation history and complete coverage across that window. Historical discount-adjusted revenue observed before evaluation is converted into SKU weights used in portfolio-level comparisons.

The final evaluation period is kept separate from the information used for sampling and historical weighting.


In [ ]:
section(9, "Chronological split and usable SKUs")

EVALUATION_WEEKS_N = int(CONFIG["evaluation_weeks"])
MIN_TRAIN_WEEKS = int(CONFIG["min_train_weeks"])

weeks = ALL_WEEKS
cutoff = EVALUATION_START

train = panel[panel["week"] < cutoff].copy()
test = panel[panel["week"] >= cutoff].copy()

train_len = train.groupby("sku")["week"].nunique()
test_len = test.groupby("sku")["week"].nunique()

SKUS = sorted([
    sku for sku in sampled_skus
    if train_len.get(sku, 0) >= MIN_TRAIN_WEEKS
    and test_len.get(sku, 0) == EVALUATION_WEEKS_N
])

panel = panel[panel["sku"].isin(SKUS)].copy()
train = train[train["sku"].isin(SKUS)].copy()
test = test[test["sku"].isin(SKUS)].copy()

print("Total calendar weeks:", len(weeks))
print("Minimum history required before final evaluation:", MIN_TRAIN_WEEKS, "weeks")
print("Final evaluation period:", cutoff.date(), "to", pd.Timestamp(EVALUATION_WEEKS[-1]).date())
print("Sampled usable SKUs:", len(SKUS), "of", sales["sku"].nunique())
print("Pre-evaluation rows:", len(train), "| Evaluation rows:", len(test))


def compute_sku_revenue_weights(history, skus, basis_label):
    """Return weights based on each SKU's share of historical discount-adjusted line revenue."""
    profile = (
        history[history["sku"].isin(skus)]
        .groupby("sku")
        .agg(
            gross_revenue=("gross_value", "sum"),
            discount_amount=("discount_amount", "sum"),
        )
        .reindex(skus, fill_value=0.0)
    )
    profile["discount_adjusted_revenue"] = (
        profile["gross_revenue"] - profile["discount_amount"]
    ).clip(lower=0.0)

    total_revenue = float(profile["discount_adjusted_revenue"].sum())
    if total_revenue > 0:
        profile["revenue_weight"] = profile["discount_adjusted_revenue"] / total_revenue
        profile["weight_fallback"] = False
    else:
        profile["revenue_weight"] = 1.0 / len(profile) if len(profile) else np.nan
        profile["weight_fallback"] = True

    profile["weight_basis"] = basis_label
    return profile.reset_index()


# Final evaluation uses only revenue observed before the evaluation cutoff.
test_revenue_profile = compute_sku_revenue_weights(
    train,
    SKUS,
    basis_label="all history before final evaluation",
)
TEST_REVENUE_WEIGHTS = test_revenue_profile.set_index("sku")["revenue_weight"].to_dict()

n_top_revenue_skus = max(1, int(np.ceil(0.20 * len(test_revenue_profile)))) if len(test_revenue_profile) else 0
top_20pct_sku_revenue_share = (
    test_revenue_profile.nlargest(n_top_revenue_skus, "discount_adjusted_revenue")["revenue_weight"].sum()
    if n_top_revenue_skus else np.nan
)
print(
    "Revenue-weighting basis: pre-evaluation discount-adjusted line revenue across eligible SKUs | "
    f"top 20% of SKUs represent {top_20pct_sku_revenue_share:.1%} of represented revenue"
)

check(
    "ABC-XYZ sampled SKU universe",
    len(SKUS) > 0,
    f"{len(SKUS)} sampled SKUs remain after pre-evaluation history and final-window checks",
    level="FAIL",
)


## 10. Demand segmentation and order-constraint settings

Summarizes each retained SKU using pre-evaluation demand statistics and assigns the demand segments used in later reporting and safety-stock fallback logic. It also translates the case-specific order-constraint setting into SKU-level batch quantities when batch rounding is enabled.


In [ ]:
section(10, "Demand segmentation and order-constraint settings")
def classify_skus(train, skus):
    # Build SKU statistics once from the pre-evaluation training panel.
    train_sub = train[train["sku"].isin(skus)].sort_values(["sku", "week"])

    stats = train_sub.groupby("sku").agg(
        train_weeks=("units", "size"),
        avg_weekly_demand=("units", "mean"),
        std_weekly_units=("units", lambda x: np.std(x)),
        zero_share=("units", lambda x: (x == 0).mean()),
        probable_stockout_weeks=("probable_stockout", "sum"),
    )
    stats["cv"] = np.where(
        stats["avg_weekly_demand"] > 0,
        stats["std_weekly_units"] / stats["avg_weekly_demand"],
        np.nan,
    )

    seasonal_corr = {}
    for sku, g in train_sub.groupby("sku")["units"]:
        h = g.values.astype(float)
        if len(h) > 52 and np.std(h[:-52]) > 0 and np.std(h[52:]) > 0:
            seasonal_corr[sku] = float(np.corrcoef(h[:-52], h[52:])[0, 1])
        else:
            seasonal_corr[sku] = 0.0
    stats["seasonal_corr_52"] = pd.Series(seasonal_corr)

    stats["segment"] = np.select(
        [stats["zero_share"] > 0.4, stats["seasonal_corr_52"] > 0.3],
        ["intermittent", "seasonal"],
        default="stable",
    )

    labels = stats["segment"].to_dict()
    details = stats.reset_index()[
        ["sku", "segment", "train_weeks", "avg_weekly_demand", "zero_share",
         "cv", "seasonal_corr_52", "probable_stockout_weeks"]
    ]
    return labels, details


sku_segment, sku_profile = classify_skus(train, SKUS)
avg_weekly_demand = sku_profile.set_index("sku")["avg_weekly_demand"].to_dict()

print("Products per segment:")
print(sku_profile["segment"].value_counts().to_string())
print("\nDemand profile summary:")
print(sku_profile.groupby("segment")[["avg_weekly_demand", "zero_share", "cv", "probable_stockout_weeks"]].mean().round(2).to_string())
sku_profile.head()

# Apply the case-specific order-constraint configuration.
# In the standardized final simulator, a positive order_batch_multiplier is
# interpreted as a minimum order expressed in weeks of that SKU's own
# pre-evaluation average demand; zero means no minimum-order constraint.
print("\nConfigured order-constraint settings:")
print(f"Reference order_batch_multiplier: {float(CONFIG['order_batch_multiplier']):.3f} weeks of SKU demand")
print("Order-constraint sensitivity grid:", CONFIG["batch_grid"])


## 11. Forecasting methods

Defines the forecasting ladder used in every case. The lower-complexity family contains naive, seasonal-naive, moving-average, ETS, and TSB methods. The ML family contains Random Forest and gradient boosting models trained across the eligible portfolio using lagged demand, rolling demand information, calendar variables, and promotion/discount information available in the constructed feature set.

The RF and GBM specifications are chosen through a lightweight tuning stage using historical origins strictly before the final evaluation period. The selected specifications are then passed to the rolling forecasting engine.


In [ ]:
section(11, "Forecasting methods")

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from joblib import Parallel, delayed, parallel_backend
import warnings


# ============================================================
# Parallel ETS settings
# ============================================================

# Number of parallel processes used for SKU-level ETS fits.
# 4 is a conservative choice that avoids excessive CPU/RAM use.
ETS_PARALLEL_JOBS = 4


# ============================================================
# Simple forecasting methods
# ============================================================

def forecast_naive(history, h):
    y = pd.Series(history).astype(float)

    last = y.iloc[-1] if len(y) else 0.0

    return np.repeat(
        max(0.0, last),
        h,
    ).astype(float)


def forecast_seasonal_naive(history, h, season=52):
    y = pd.Series(history).astype(float)

    if len(y) >= season:
        last_cycle = y.iloc[-season:].values.astype(float)

        return np.array(
            [
                max(0.0, last_cycle[i % season])
                for i in range(h)
            ],
            dtype=float,
        )

    return forecast_naive(y, h)


def forecast_moving_average(history, h, window=8):
    y = pd.Series(history).astype(float)

    avg = (
        y.iloc[-window:].mean()
        if len(y)
        else 0.0
    )

    return np.repeat(
        max(0.0, avg),
        h,
    ).astype(float)


def forecast_ets(history, h):
    """
    Exponential-smoothing forecast.

    Highly intermittent series are fitted without a trend.
    Annual seasonality is attempted only when at least two
    complete 52-week seasons are available.

    If ETS cannot be estimated for a particular SKU/history,
    the function falls back to the moving-average forecast.
    """

    y = (
        pd.Series(history)
        .astype(float)
        .values
        .astype(float)
    )

    # Very short or constant histories do not justify
    # estimating an ETS model.
    if len(y) < 4 or np.all(y == y[0]):
        return forecast_moving_average(y, h)

    try:
        # Avoid fitting a trend to highly intermittent demand.
        zero_share = float(np.mean(y == 0))

        trend_component = (
            None
            if zero_share > 0.4
            else "add"
        )

        # Annual seasonal ETS only if at least two complete
        # annual cycles are available.
        if len(y) >= 104:
            model = ExponentialSmoothing(
                y,
                trend=trend_component,
                seasonal="add",
                seasonal_periods=52,
            )

        else:
            model = ExponentialSmoothing(
                y,
                trend=trend_component,
                seasonal=None,
            )

        # Ignore convergence warnings here.
        # Genuine exceptions still trigger the fallback below.
        with warnings.catch_warnings():
            warnings.simplefilter(
                "ignore",
                ConvergenceWarning,
            )

            fit = model.fit(
                optimized=True,
                use_brute=False,
            )

        return np.clip(
            np.asarray(
                fit.forecast(h),
                dtype=float,
            ),
            0,
            None,
        )

    except Exception:
        return forecast_moving_average(y, h)


def forecast_tsb(history, h, alpha=0.2, beta=0.2):
    """
    Teunter-Syntetos-Babai forecast for intermittent demand.

    TSB separately smooths:
    1. the probability of demand occurrence;
    2. the demand size conditional on occurrence.

    Forecast = occurrence probability x demand size.
    """

    y = np.asarray(
        history,
        dtype=float,
    )

    if len(y) == 0:
        return np.zeros(
            h,
            dtype=float,
        )

    occurrence = (
        y > 0
    ).astype(float)

    positive = y[y > 0]

    p = (
        occurrence[:min(4, len(occurrence))].mean()
        if len(occurrence)
        else 0.0
    )

    z = (
        positive[:min(4, len(positive))].mean()
        if len(positive)
        else 0.0
    )

    for val, occ in zip(
        y,
        occurrence,
    ):
        p = p + beta * (occ - p)

        if occ > 0:
            z = z + alpha * (val - z)

    fc = max(
        0.0,
        p * z,
    )

    return np.repeat(
        fc,
        h,
    ).astype(float)


SIMPLE_FNS = {
    "naive": lambda h, H: forecast_naive(
        h,
        H,
    ),

    "seasonal_naive": lambda h, H: forecast_seasonal_naive(
        h,
        H,
        season=52,
    ),

    "moving_avg": lambda h, H: forecast_moving_average(
        h,
        H,
        window=8,
    ),

    "ets": lambda h, H: forecast_ets(
        h,
        H,
    ),

    "tsb": lambda h, H: forecast_tsb(
        h,
        H,
    ),
}

SIMPLE_METHODS = list(
    SIMPLE_FNS.keys()
)


# ============================================================
# Machine-learning feature engineering
# ============================================================

def make_features(panel):
    df = (
        panel
        .sort_values(
            ["sku", "week"]
        )
        .copy()
    )

    g = df.groupby(
        "sku"
    )["units"]

    for lag in [1, 2, 4]:
        df[f"lag_{lag}"] = g.shift(
            lag
        )

    df["roll_mean_4"] = g.transform(
        lambda x: (
            x.shift(1)
            .rolling(
                4,
                min_periods=1,
            )
            .mean()
        )
    )

    df["roll_std_4"] = g.transform(
        lambda x: (
            x.shift(1)
            .rolling(
                4,
                min_periods=2,
            )
            .std()
        )
    )

    df["month"] = (
        df["week"]
        .dt.month
        .astype(int)
    )

    df["weekofyear"] = (
        df["week"]
        .dt.isocalendar()
        .week
        .astype(int)
    )

    # Retained descriptively in the feature table,
    # but not used by the ML models because avg_discount
    # already captures the promotion information.
    df["on_promo"] = (
        df["on_promo"]
        .astype(int)
    )

    df["avg_discount"] = (
        df["avg_discount"]
        .astype(float)
    )

    return df


feat = make_features(
    panel
)


FEATURE_COLS = [
    "lag_1",
    "lag_2",
    "lag_4",
    "roll_mean_4",
    "roll_std_4",
    "month",
    "weekofyear",
    "avg_discount",
]


# ============================================================
# Machine-learning models: lightweight pre-evaluation tuning
# ============================================================
#
# The forecasting/inventory logic below is unchanged. This block only selects
# one RF specification and one GBM specification using four leakage-safe
# historical origins strictly before the final evaluation window. The selected
# factories are then consumed by the existing rolling and direct-H engines.
#
# Candidate sets are intentionally small to keep runtime modest and to avoid
# post-hoc model hunting.

RF_CANDIDATES = {
    "rf_flexible": {
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_leaf": 2,
        "max_features": 0.7,
    },
    "rf_current": {
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_leaf": 5,
        "max_features": 0.7,
    },
    "rf_smooth": {
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_leaf": 10,
        "max_features": 0.7,
    },
}

GBM_CANDIDATES = {
    "gbm_smooth": {
        "max_iter": 200,
        "learning_rate": 0.05,
        "max_depth": 4,
        "min_samples_leaf": 30,
        "l2_regularization": 0.1,
    },
    "gbm_current": {
        "max_iter": 200,
        "learning_rate": 0.05,
        "max_depth": 6,
        "min_samples_leaf": 20,
        "l2_regularization": 0.1,
    },
    "gbm_flexible": {
        "max_iter": 200,
        "learning_rate": 0.05,
        "max_depth": 8,
        "min_samples_leaf": 10,
        "l2_regularization": 0.1,
    },
}

# Original fixed specifications are retained as a safe fallback.
DEFAULT_RF_PARAMS = RF_CANDIDATES["rf_current"].copy()
DEFAULT_GBM_PARAMS = GBM_CANDIDATES["gbm_current"].copy()


def _make_tuning_model(family, params):
    if family == "rf":
        return RandomForestRegressor(
            **params,
            n_jobs=int(CONFIG.get("ml_n_jobs", 4)),
            random_state=0,
        )
    if family == "gbm":
        return HistGradientBoostingRegressor(
            **params,
            early_stopping=False,
            random_state=0,
        )
    raise ValueError(f"Unknown ML family: {family}")


def _tuning_wape(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    denom = float(np.sum(np.abs(actual)))
    if denom <= 0:
        return np.nan
    return float(np.sum(np.abs(actual - forecast)) / denom)


def _weighted_average(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights >= 0)
    if mask.sum() == 0 or float(weights[mask].sum()) <= 0:
        return np.nan
    return float(np.average(values[mask], weights=weights[mask]))


def _choose_tuning_origins():
    """Choose four evenly spaced origins from the final 12 pre-evaluation weeks."""
    pre_eval_weeks = pd.DatetimeIndex(sorted(pd.to_datetime(train["week"].unique())))
    if len(pre_eval_weeks) < 4:
        return list(pre_eval_weeks)

    # Use at most the last 12 pre-evaluation weeks, then spread four origins
    # across that window. All selected origins are strictly before cutoff.
    window = pre_eval_weeks[-min(12, len(pre_eval_weeks)):]
    idx = np.linspace(0, len(window) - 1, num=min(4, len(window)), dtype=int)
    origins = list(pd.DatetimeIndex(window[idx]).unique())
    return origins


TUNING_ORIGINS = _choose_tuning_origins()

# Fix economic weights before the earliest tuning origin so the candidate
# comparison does not use revenue information from later tuning target weeks.
if TUNING_ORIGINS:
    tuning_weight_cutoff = min(TUNING_ORIGINS)
    tuning_weight_history = train[train["week"] < tuning_weight_cutoff].copy()
else:
    tuning_weight_cutoff = cutoff
    tuning_weight_history = train.copy()

TUNING_REVENUE_PROFILE = compute_sku_revenue_weights(
    tuning_weight_history,
    SKUS,
    basis_label="history strictly before first ML tuning origin",
)
TUNING_REVENUE_WEIGHTS = (
    TUNING_REVENUE_PROFILE.set_index("sku")["revenue_weight"].to_dict()
)


def evaluate_ml_candidate(family, candidate_name, params, tuning_origins):
    """
    Evaluate one ML specification on four historical one-step origins.

    Selection metric matches the notebook's headline accuracy logic:
    first compute WAPE per SKU across the tuning origins, then take the
    revenue-weighted mean across SKUs. No final-evaluation week is used.
    """
    prediction_rows = []
    origin_rows_out = []

    for origin_week in tuning_origins:
        training_feat = feat[feat["week"] < origin_week].copy()
        training_feat = training_feat.dropna(subset=["lag_1"])

        target_feat = feat[
            (feat["week"] == origin_week)
            & (feat["sku"].isin(SKUS))
        ].copy()
        target_feat = target_feat.dropna(subset=["lag_1"])

        if training_feat.empty or target_feat.empty:
            origin_rows_out.append({
                "family": family,
                "candidate": candidate_name,
                "origin_week": pd.Timestamp(origin_week),
                "origin_wape": np.nan,
                "n_target_skus": 0,
                "fit_ok": False,
                "message": "empty training or target rows",
            })
            continue

        try:
            X_train = training_feat[FEATURE_COLS].reindex(columns=FEATURE_COLS).fillna(0.0)
            y_train = training_feat["units"].astype(float)
            X_target = target_feat[FEATURE_COLS].reindex(columns=FEATURE_COLS).fillna(0.0)

            model = _make_tuning_model(family, params)
            model.fit(X_train, y_train)
            pred = np.clip(model.predict(X_target), 0, None)

            actual = target_feat["units"].to_numpy(dtype=float)
            origin_wape = _tuning_wape(actual, pred)

            origin_rows_out.append({
                "family": family,
                "candidate": candidate_name,
                "origin_week": pd.Timestamp(origin_week),
                "origin_wape": origin_wape,
                "n_target_skus": int(len(target_feat)),
                "fit_ok": True,
                "message": "",
            })

            prediction_rows.extend({
                "sku": sku,
                "origin_week": pd.Timestamp(origin_week),
                "actual": float(y),
                "forecast": float(p),
            } for sku, y, p in zip(target_feat["sku"], actual, pred))

        except Exception as exc:
            origin_rows_out.append({
                "family": family,
                "candidate": candidate_name,
                "origin_week": pd.Timestamp(origin_week),
                "origin_wape": np.nan,
                "n_target_skus": int(len(target_feat)),
                "fit_ok": False,
                "message": f"{type(exc).__name__}: {exc}",
            })

    pred_df = pd.DataFrame(prediction_rows)
    origin_df = pd.DataFrame(origin_rows_out)

    if pred_df.empty:
        return np.nan, 0, 0.0, origin_df

    sku_rows = []
    for sku, g in pred_df.groupby("sku"):
        score = _tuning_wape(g["actual"].values, g["forecast"].values)
        if np.isfinite(score):
            sku_rows.append({
                "sku": sku,
                "wape": score,
                "revenue_weight": float(TUNING_REVENUE_WEIGHTS.get(sku, 0.0)),
            })

    sku_scores = pd.DataFrame(sku_rows)
    if sku_scores.empty:
        return np.nan, 0, 0.0, origin_df

    score = _weighted_average(
        sku_scores["wape"].values,
        sku_scores["revenue_weight"].values,
    )
    revenue_covered = float(sku_scores["revenue_weight"].sum())
    return score, int(len(sku_scores)), revenue_covered, origin_df


def select_ml_configuration():
    rows = []
    origin_details = []

    candidate_sets = [
        ("rf", RF_CANDIDATES),
        ("gbm", GBM_CANDIDATES),
    ]

    for family, candidates in candidate_sets:
        for candidate_name, params in candidates.items():
            score, n_skus, revenue_covered, origin_df = evaluate_ml_candidate(
                family,
                candidate_name,
                params,
                TUNING_ORIGINS,
            )
            rows.append({
                "family": family,
                "candidate": candidate_name,
                "revenue_weighted_wape": score,
                "n_skus": n_skus,
                "revenue_weight_covered": revenue_covered,
                "params": params.copy(),
            })
            if not origin_df.empty:
                origin_details.append(origin_df)

    results = pd.DataFrame(rows)
    origin_results = (
        pd.concat(origin_details, ignore_index=True)
        if origin_details else
        pd.DataFrame()
    )

    selected = {}
    for family, default_params in [
        ("rf", DEFAULT_RF_PARAMS),
        ("gbm", DEFAULT_GBM_PARAMS),
    ]:
        valid = results[
            (results["family"] == family)
            & np.isfinite(results["revenue_weighted_wape"])
        ].sort_values("revenue_weighted_wape")

        if valid.empty:
            print(
                f"[WARN] ML tuning produced no valid {family.upper()} candidate. "
                "Falling back to the original fixed specification."
            )
            selected[family] = default_params.copy()
        else:
            selected[family] = valid.iloc[0]["params"].copy()

    return results, origin_results, selected["rf"], selected["gbm"]


print("\nLightweight ML tuning origins:")
print([pd.Timestamp(w).date() for w in TUNING_ORIGINS])
print(
    "Tuning weight basis cutoff:",
    pd.Timestamp(tuning_weight_cutoff).date() if len(TUNING_ORIGINS) else "n/a",
)
print("Final evaluation starts:", pd.Timestamp(cutoff).date())

ML_TUNING_RESULTS, ML_TUNING_ORIGIN_RESULTS, BEST_RF_PARAMS, BEST_GBM_PARAMS = (
    select_ml_configuration()
)

print("\nML tuning results (lower revenue-weighted WAPE is better):")
if ML_TUNING_RESULTS.empty:
    print("No tuning results available; original specifications will be used.")
else:
    display_cols = [
        "family", "candidate", "revenue_weighted_wape",
        "n_skus", "revenue_weight_covered",
    ]
    print(
        ML_TUNING_RESULTS[display_cols]
        .sort_values(["family", "revenue_weighted_wape"])
        .round(4)
        .to_string(index=False)
    )

print("\nSelected RF parameters:")
print(BEST_RF_PARAMS)
print("Selected GBM parameters:")
print(BEST_GBM_PARAMS)

print("\nSelected-vs-current tuning score check:")
for family, current_name in [("rf", "rf_current"), ("gbm", "gbm_current")]:
    fam = ML_TUNING_RESULTS[ML_TUNING_RESULTS["family"] == family].copy()
    valid = fam[np.isfinite(fam["revenue_weighted_wape"])].sort_values("revenue_weighted_wape")
    current = fam[fam["candidate"] == current_name]
    if valid.empty or current.empty or not np.isfinite(current.iloc[0]["revenue_weighted_wape"]):
        print(f"{family.upper()}: comparison unavailable")
        continue
    best = valid.iloc[0]
    current_score = float(current.iloc[0]["revenue_weighted_wape"])
    best_score = float(best["revenue_weighted_wape"])
    improvement = 100.0 * (current_score - best_score) / current_score if current_score > 0 else np.nan
    print(
        f"{family.upper()}: selected {best['candidate']} | "
        f"current={current_score:.4f}, selected={best_score:.4f}, "
        f"WAPE improvement={improvement:.2f}%"
    )

# Compact tuning diagnostics: these are reporting checks only and do not alter
# model selection or downstream forecasting logic.
print("\nML tuning diagnostic by origin:")
if ML_TUNING_ORIGIN_RESULTS.empty:
    print("No origin-level tuning diagnostics available.")
else:
    origin_diag = (
        ML_TUNING_ORIGIN_RESULTS
        .groupby(["family", "candidate"], as_index=False)
        .agg(
            successful_origins=("fit_ok", "sum"),
            mean_origin_wape=("origin_wape", "mean"),
            max_origin_wape=("origin_wape", "max"),
            mean_target_skus=("n_target_skus", "mean"),
        )
    )
    print(origin_diag.round(4).to_string(index=False))

    # Stability diagnostic: count how many tuning origins each candidate wins
    # within its model family. This uses already-computed origin scores only.
    valid_origin_scores = ML_TUNING_ORIGIN_RESULTS[
        ML_TUNING_ORIGIN_RESULTS["fit_ok"]
        & np.isfinite(ML_TUNING_ORIGIN_RESULTS["origin_wape"])
    ].copy()

    if not valid_origin_scores.empty:
        winners = (
            valid_origin_scores
            .sort_values(["family", "origin_week", "origin_wape", "candidate"])
            .groupby(["family", "origin_week"], as_index=False)
            .first()[["family", "origin_week", "candidate", "origin_wape"]]
        )
        win_counts = (
            winners.groupby(["family", "candidate"])
            .size()
            .rename("origin_wins")
            .reset_index()
        )
        origin_counts = (
            winners.groupby("family")["origin_week"]
            .nunique()
            .rename("origins_compared")
            .reset_index()
        )
        win_counts = win_counts.merge(origin_counts, on="family", how="left")
        print("\nML tuning stability — best candidate by origin:")
        print(win_counts.to_string(index=False))
        print("\nOrigin-level winners:")
        print(winners.to_string(index=False))

# Guardrails for a company-run notebook: warn visibly if the lightweight tuning
# stage is not fully usable, but retain the original model specification as a
# fallback so the rest of the notebook can still run.
if any(pd.Timestamp(w) >= pd.Timestamp(cutoff) for w in TUNING_ORIGINS):
    print("[WARN] At least one tuning origin is not strictly before final evaluation.")
else:
    print("[PASS] All ML tuning origins are strictly before final evaluation.")

for family in ["rf", "gbm"]:
    fam = ML_TUNING_RESULTS[ML_TUNING_RESULTS["family"] == family]
    n_valid = int(np.isfinite(fam["revenue_weighted_wape"]).sum()) if len(fam) else 0
    if n_valid == 3:
        print(f"[PASS] {family.upper()} tuning: all 3 candidate specifications produced valid scores.")
    else:
        print(f"[WARN] {family.upper()} tuning: only {n_valid} of 3 candidate specifications produced valid scores.")


def get_ml_factories():
    return {
        "ml_gbm": lambda: HistGradientBoostingRegressor(
            **BEST_GBM_PARAMS,
            early_stopping=False,
            random_state=0,
        ),
        "ml_rf": lambda: RandomForestRegressor(
            **BEST_RF_PARAMS,
            n_jobs=int(CONFIG.get("ml_n_jobs", 4)),
            random_state=0,
        ),
    }


ML_FACTORIES = get_ml_factories()


ML_METHODS = list(
    ML_FACTORIES.keys()
)

ALL_METHODS = (
    SIMPLE_METHODS
    + ML_METHODS
)


def prep_X(X):
    return (
        X
        .reindex(
            columns=FEATURE_COLS
        )
        .fillna(0.0)
    )


print(
    "Simple methods:",
    SIMPLE_METHODS,
)

print(
    "ML methods:",
    ML_METHODS,
)

## 12. Rolling weekly re-forecasting engine

Generates forecasts as they would have been produced sequentially through time. At each origin, only information available up to that point is used. Simple methods are updated from the SKU history, while the standardized ML models are trained one week ahead and extended recursively to the maximum protection horizon required by the scenario grid.

The resulting forecast cache is the common input for forecast evaluation, safety-stock calibration, and the inventory simulation.


In [ ]:
section(12, "Rolling weekly re-forecasting engine")
PANEL_SORTED = panel.sort_values(["sku", "week"])
SKU_HISTORY = {
    sku: {"weeks": g["week"].values, "units": g["units"].values.astype(float)}
    for sku, g in PANEL_SORTED.groupby("sku")
}
PANEL_PROMO = panel.set_index(["sku", "week"])[["on_promo", "avg_discount"]].to_dict("index")


def get_future_calendar_rows(sku, origin_week, horizon):
    weeks_future = pd.date_range(pd.Timestamp(origin_week), periods=horizon, freq="W-MON")
    on_promo_list, discount_list = [], []
    for w in weeks_future:
        info = PANEL_PROMO.get((sku, w))
        if info is None:
            on_promo_list.append(False)
            discount_list.append(0.0)
        else:
            on_promo_list.append(bool(info["on_promo"]))
            discount_list.append(float(info["avg_discount"]))
    return pd.DataFrame({"sku": sku, "week": weeks_future, "on_promo": on_promo_list, "avg_discount": discount_list})


def build_one_feature_row(history_units, week_date, on_promo, avg_discount):
    # Return a lightweight feature record for recursive multi-step prediction.
    h = np.asarray(history_units, dtype=float)
    return {
        "lag_1": h[-1] if len(h) >= 1 else 0.0,
        "lag_2": h[-2] if len(h) >= 2 else 0.0,
        "lag_4": h[-4] if len(h) >= 4 else 0.0,
        "roll_mean_4": h[-4:].mean() if len(h) >= 1 else 0.0,
        "roll_std_4": h[-4:].std(ddof=0) if len(h) >= 2 else 0.0,
        "month": int(pd.Timestamp(week_date).month),
        "weekofyear": int(pd.Timestamp(week_date).isocalendar().week),
        "on_promo": int(on_promo),
        "avg_discount": float(avg_discount),
    }


def fit_ml_models(origin_week):
    # Trains on the full pooled history available at this origin -- no row
    # cap/subsampling, by request: every SKU's complete available history
    # contributes to each fit. The .isin(SKUS) check that used to be here was
    # redundant: `feat` is built from `panel`, which was already restricted to
    # SKUS back in Block 6 -- every row already satisfies it by construction.
    training_feat = feat[feat["week"] < origin_week].copy()
    training_feat = training_feat.dropna(subset=["lag_1"])
    if len(training_feat) == 0:
        return {}
    X_train = prep_X(training_feat[FEATURE_COLS])
    y_train = training_feat["units"].astype(float)
    fitted = {}
    for name, make_model in ML_FACTORIES.items():
        try:
            fitted[name] = make_model().fit(X_train, y_train)
        except Exception as exc:
            print(
                f"Warning: ML method '{name}' failed to fit at origin week "
                f"{pd.Timestamp(origin_week).date()} ({type(exc).__name__}: {exc}). "
                "Skipping this method for this origin week only."
            )
    return fitted



def forecast_at_origin(origin_week, horizon, fitted_ml=None, include_ml=True):

    if include_ml and fitted_ml is None:
        fitted_ml = fit_ml_models(origin_week)
    elif not include_ml:
        fitted_ml = {}

    rows = []

    sku_hist_map = {}
    sku_future_map = {}

    # ---------------------------------------------------------
    # 1. Build SKU contexts once
    # ---------------------------------------------------------
    sku_contexts = []

    for sku in SKUS:

        sku_data = SKU_HISTORY.get(sku)

        if sku_data is None:
            continue

        idx = np.searchsorted(
            sku_data["weeks"],
            np.datetime64(origin_week),
        )

        hist = sku_data["units"][:idx]

        if len(hist) == 0:
            continue

        future_rows = get_future_calendar_rows(
            sku,
            origin_week,
            horizon,
        )

        sku_contexts.append(
            (
                sku,
                hist,
                future_rows,
            )
        )

        if include_ml and fitted_ml:
            sku_hist_map[sku] = list(hist)

            sku_future_map[sku] = (
                future_rows
                .sort_values("week")
                .reset_index(drop=True)
            )

    # ---------------------------------------------------------
    # 2. Fit ETS forecasts in parallel across SKUs
    # ---------------------------------------------------------
    #
    # ETS is by far the expensive simple method.
    # Naive, seasonal naive, moving average and TSB are cheap,
    # so keeping those sequential avoids unnecessary overhead.
    #
    # joblib returns results in the same order as sku_contexts.
    # Therefore this does not alter the analytical result.
    # ---------------------------------------------------------

    if sku_contexts:

        with parallel_backend(
            "loky",
            inner_max_num_threads=1,
        ):
            ets_results = Parallel(
                n_jobs=ETS_PARALLEL_JOBS,
            )(
                delayed(forecast_ets)(
                    hist,
                    horizon,
                )
                for sku, hist, future_rows in sku_contexts
            )

        ets_forecasts = {
            sku_contexts[i][0]: np.clip(
                np.asarray(ets_results[i], dtype=float),
                0,
                None,
            )
            for i in range(len(sku_contexts))
        }

    else:
        ets_forecasts = {}

    # ---------------------------------------------------------
    # 3. Generate all simple-method forecast rows
    # ---------------------------------------------------------

    for sku, hist, future_rows in sku_contexts:

        for method_name, fn in SIMPLE_FNS.items():

            # ETS was already calculated in parallel.
            if method_name == "ets":
                fc = ets_forecasts[sku]

            else:
                fc = np.clip(
                    fn(
                        pd.Series(hist),
                        horizon,
                    ),
                    0,
                    None,
                )

            for k, pred in enumerate(fc, start=1):

                rows.append(
                    {
                        "origin_week": origin_week,
                        "target_week": future_rows.iloc[k - 1]["week"],
                        "horizon_step": k,
                        "sku": sku,
                        "method": method_name,
                        "forecast": float(pred),
                    }
                )

    # ---------------------------------------------------------
    # 4. Recursive ML forecasting
    # ---------------------------------------------------------
    #
    # This part is analytically identical to the current
    # implementation.
    # ---------------------------------------------------------

    if include_ml and fitted_ml and sku_hist_map:

        eligible_skus = list(
            sku_hist_map.keys()
        )

        for method_name, model in fitted_ml.items():

            running_hist = {
                sku: list(sku_hist_map[sku])
                for sku in eligible_skus
            }

            step_predictions = {
                sku: []
                for sku in eligible_skus
            }

            failed_skus = set()

            for step in range(horizon):

                feature_rows = []
                step_skus = []

                for sku in eligible_skus:

                    if sku in failed_skus:
                        continue

                    future_rows = sku_future_map[sku]

                    if step >= len(future_rows):
                        continue

                    row = future_rows.iloc[step]

                    feature_rows.append(
                        build_one_feature_row(
                            running_hist[sku],
                            row["week"],
                            row["on_promo"],
                            row["avg_discount"],
                        )
                    )

                    step_skus.append(sku)

                if not feature_rows:
                    continue

                X_batch = pd.DataFrame(
                    feature_rows
                )[FEATURE_COLS]

                try:

                    preds = np.clip(
                        model.predict(
                            prep_X(X_batch)
                        ),
                        0,
                        None,
                    )

                except Exception:

                    preds = []
                    kept_skus = []

                    for i, sku in enumerate(step_skus):

                        try:

                            X_single = pd.DataFrame(
                                [feature_rows[i]]
                            )[FEATURE_COLS]

                            pred = float(
                                np.clip(
                                    model.predict(
                                        prep_X(X_single)
                                    )[0],
                                    0,
                                    None,
                                )
                            )

                            preds.append(pred)
                            kept_skus.append(sku)

                        except Exception as exc:

                            print(
                                f"Warning: ML method '{method_name}' "
                                f"failed to forecast SKU '{sku}' "
                                f"at step {step + 1}, origin "
                                f"{pd.Timestamp(origin_week).date()} "
                                f"({type(exc).__name__}: {exc}). "
                                "Skipping this SKU/method/origin "
                                "combination only."
                            )

                            failed_skus.add(sku)

                    step_skus = kept_skus
                    preds = np.array(preds)

                for sku, pred in zip(
                    step_skus,
                    preds,
                ):

                    running_hist[sku].append(
                        float(pred)
                    )

                    step_predictions[sku].append(
                        float(pred)
                    )

            for sku in eligible_skus:

                future_rows = sku_future_map[sku]

                for k, pred in enumerate(
                    step_predictions[sku],
                    start=1,
                ):

                    rows.append(
                        {
                            "origin_week": origin_week,
                            "target_week": future_rows.iloc[k - 1]["week"],
                            "horizon_step": k,
                            "sku": sku,
                            "method": method_name,
                            "forecast": pred,
                        }
                    )

    return pd.DataFrame(rows)


REFIT_EVERY_N_ORIGINS = int(CONFIG.get("refit_every_n_origins", 3))  # Refit ML models at the configured cadence,
                            # reusing the most recent fit in between (still trained on ALL
                            # available data each time it refits -- no row cap). Verified:
                            # ~2.6% relative prediction difference, worst case -- small next
                            # to the ~3-4% real gap between ets and the ML
                            # methods in the current case results.


def build_rolling_forecast_cache(origin_weeks, horizon, include_ml=True):
    parts = []
    n = len(origin_weeks)
    cached_fitted_ml = None
    for i, ow in enumerate(origin_weeks, start=1):
        t0 = time.time()
        refit_now = include_ml and (cached_fitted_ml is None or (i - 1) % REFIT_EVERY_N_ORIGINS == 0)
        if refit_now:
            cached_fitted_ml = fit_ml_models(ow)
        parts.append(forecast_at_origin(ow, horizon, fitted_ml=cached_fitted_ml, include_ml=include_ml))
        elapsed = time.time() - t0
        tag = " [refit]" if refit_now else " [reused model]"
        print(f"  Origin week {i}/{n} ({pd.Timestamp(ow).date()}) done in {elapsed:.1f}s{tag}")
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


test_weeks = list(np.sort(test["week"].unique()))
rolling_test_forecasts = build_rolling_forecast_cache(test_weeks, MAX_PROTECTION_HORIZON, include_ml=True)
print("Rolling test forecast rows:", len(rolling_test_forecasts))

rolling_test_forecasts.head()


## 13. Out-of-sample forecast accuracy

Evaluates the rolling forecasts on the final out-of-sample window. One-week performance is summarized with WAPE and MASE, while cumulative protection-period WAPE evaluates the horizon relevant to replenishment decisions. Portfolio results are reported with demand and historical-revenue weighting.

Headline simple-versus-ML comparisons are restricted to a common valid SKU sample and must meet the configured minimum revenue-coverage threshold, so differences are not driven by methods being evaluated on different subsets of the portfolio.


In [ ]:
section(13, "Out-of-sample forecast accuracy")
def wape(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    denom = np.sum(np.abs(actual))
    return np.nan if denom == 0 else np.sum(np.abs(actual - forecast)) / denom


def mase(actual, forecast, train_history, season=1):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    train_history = np.asarray(train_history, dtype=float)
    if len(train_history) <= season:
        return np.nan
    denom = np.mean(np.abs(train_history[season:] - train_history[:-season]))
    if denom == 0:
        return np.nan
    return np.mean(np.abs(actual - forecast)) / denom


def weighted_mean(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights >= 0)
    if mask.sum() == 0 or weights[mask].sum() == 0:
        return np.nan
    return float(np.average(values[mask], weights=weights[mask]))


def weighted_corr(x, y, weights):
    """Weighted Pearson correlation, used only as a descriptive diagnostic."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(weights) & (weights >= 0)
    if mask.sum() < 2 or weights[mask].sum() == 0:
        return np.nan
    x, y, weights = x[mask], y[mask], weights[mask]
    weights = weights / weights.sum()
    x_bar = np.sum(weights * x)
    y_bar = np.sum(weights * y)
    cov = np.sum(weights * (x - x_bar) * (y - y_bar))
    var_x = np.sum(weights * (x - x_bar) ** 2)
    var_y = np.sum(weights * (y - y_bar) ** 2)
    if var_x <= 0 or var_y <= 0:
        return np.nan
    return float(cov / np.sqrt(var_x * var_y))


MIN_COMMON_REVENUE_COVERAGE = float(CONFIG.get("min_common_revenue_coverage", 0.80))


def restrict_to_common_sku_sample(
    df,
    methods,
    metric_col,
    weight_col="revenue_weight",
    threshold=MIN_COMMON_REVENUE_COVERAGE,
    context="comparison",
):
    """Restrict methods to the same valid SKU set and enforce revenue coverage.

    The returned rows contain only SKUs for which every requested method has a
    finite value for `metric_col`. Revenue weights remain the original portfolio
    weights; weighted_mean renormalizes them within the retained common sample.
    """
    methods = list(dict.fromkeys(methods))
    present_methods = set(df["method"].dropna().unique())
    missing_methods = [m for m in methods if m not in present_methods]
    if missing_methods:
        raise ValueError(f"{context}: missing method(s) required for comparison: {missing_methods}")

    requested = df[df["method"].isin(methods)].copy()
    valid = requested[np.isfinite(pd.to_numeric(requested[metric_col], errors="coerce"))].copy()
    method_counts = valid.groupby("sku")["method"].nunique()
    common_skus = method_counts[method_counts == len(methods)].index

    sku_weights = requested.groupby("sku")[weight_col].max()
    common_revenue_coverage = float(sku_weights.reindex(common_skus).fillna(0.0).sum())
    metadata = {
        "context": context,
        "metric": metric_col,
        "methods": ", ".join(methods),
        "common_n_skus": int(len(common_skus)),
        "common_revenue_coverage": common_revenue_coverage,
        "minimum_required_coverage": float(threshold),
    }

    if len(common_skus) == 0 or common_revenue_coverage < threshold:
        raise ValueError(
            f"{context}: controlled common-SKU comparison is not reliable. "
            f"Common SKUs={len(common_skus)}, represented revenue="
            f"{common_revenue_coverage:.1%}, required at least {threshold:.0%}."
        )

    out = requested[requested["sku"].isin(common_skus)].copy()
    out["common_revenue_coverage"] = common_revenue_coverage
    out["common_n_skus"] = int(len(common_skus))
    return out, metadata


def score_one_step_rolling(forecast_cache):
    one = forecast_cache[forecast_cache["horizon_step"] == 1].copy()
    actual_lookup = panel.set_index(["sku", "week"])["units"].to_dict()
    one["actual"] = [actual_lookup.get((r.sku, r.target_week), np.nan) for r in one.itertuples()]
    one = one.dropna(subset=["actual"])

    hist_by_sku = {
        sku: g["units"].values.astype(float)
        for sku, g in train.sort_values(["sku", "week"]).groupby("sku")
    }

    rows = []
    for (sku, method_name), g in one.groupby(["sku", "method"]):
        g = g.sort_values("origin_week")
        actual = g["actual"].values.astype(float)
        fc = g["forecast"].values.astype(float)
        hist = hist_by_sku.get(sku, np.array([]))
        rows.append({
            "sku": sku,
            "method": method_name,
            "segment": sku_segment.get(sku, "unknown"),
            "actual_units": actual.sum(),
            "wape": wape(actual, fc),
            "mase": mase(actual, fc, hist, season=1),
        })
    return pd.DataFrame(rows)


def score_protection_period_rolling(forecast_cache, horizon):
    """Score cumulative forecast accuracy over the inventory protection period."""
    if forecast_cache is None or len(forecast_cache) == 0:
        return pd.DataFrame(columns=["sku", "method", "segment", "actual_units", "protection_wape"])

    actual_lookup = panel.set_index(["sku", "week"])["units"].to_dict()
    sub = forecast_cache[forecast_cache["horizon_step"] <= horizon].copy()
    rows = []

    for (sku, method_name, origin_week), g in sub.groupby(["sku", "method", "origin_week"]):
        g = g.sort_values("horizon_step")
        if len(g) < horizon:
            continue
        actual = np.array([actual_lookup.get((sku, wk), np.nan) for wk in g["target_week"]], dtype=float)
        if np.isnan(actual[:horizon]).any():
            continue
        rows.append({
            "sku": sku,
            "method": method_name,
            "segment": sku_segment.get(sku, "unknown"),
            "origin_week": origin_week,
            "actual_sum": float(actual[:horizon].sum()),
            "forecast_sum": float(g["forecast"].values[:horizon].sum()),
        })

    origin_scores = pd.DataFrame(rows)
    if origin_scores.empty:
        return pd.DataFrame(columns=["sku", "method", "segment", "actual_units", "protection_wape"])

    out = []
    for (sku, method_name), g in origin_scores.groupby(["sku", "method"]):
        out.append({
            "sku": sku,
            "method": method_name,
            "segment": sku_segment.get(sku, "unknown"),
            "actual_units": float(g["actual_sum"].sum()),
            "protection_wape": wape(g["actual_sum"].values, g["forecast_sum"].values),
        })
    return pd.DataFrame(out)


scores_all_available = score_one_step_rolling(rolling_test_forecasts)
scores_all_available["revenue_weight"] = scores_all_available["sku"].map(TEST_REVENUE_WEIGHTS).fillna(0.0)
# Full diagnostic scoreboard: each method is reported on its OWN available
# SKUs (no cross-method common-sample requirement). This is descriptive only
# -- it is not used for method selection or for the headline ML-vs-simple
# comparison, so one method's occasional missing forecast should not remove
# SKUs from another method's reported accuracy.
scores = scores_all_available
summary = (
    scores.dropna(subset=["wape"])
    .groupby("method")
    .apply(lambda g: pd.Series({
        "weighted_wape": weighted_mean(g["wape"], g["actual_units"]),
        "revenue_weighted_wape": weighted_mean(g["wape"], g["revenue_weight"]),
        "median_sku_wape": g["wape"].median(),
        "median_mase": g["mase"].median(),
        "revenue_weight_covered": g.loc[g["wape"].notna(), "revenue_weight"].sum(),
        "n_skus": len(g),
    }))
    .sort_values("revenue_weighted_wape")
)

seg_summary = (
    scores.dropna(subset=["wape"])
    .groupby(["segment", "method"])
    .apply(lambda g: weighted_mean(g["wape"], g["actual_units"]))
    .unstack("method")
)
seg_summary = seg_summary.reindex(columns=[m for m in ALL_METHODS if m in seg_summary.columns])

revenue_seg_summary = (
    scores.dropna(subset=["wape"])
    .groupby(["segment", "method"])
    .apply(lambda g: weighted_mean(g["wape"], g["revenue_weight"]))
    .unstack("method")
)
revenue_seg_summary = revenue_seg_summary.reindex(columns=[m for m in ALL_METHODS if m in revenue_seg_summary.columns])

base_accuracy_horizon = int(CONFIG["lead_time_weeks"]) + int(CONFIG["review_period_weeks"])
protection_scores_all_available = score_protection_period_rolling(
    rolling_test_forecasts,
    base_accuracy_horizon,
)
protection_scores_all_available["revenue_weight"] = (
    protection_scores_all_available["sku"].map(TEST_REVENUE_WEIGHTS).fillna(0.0)
)
protection_scores = protection_scores_all_available

protection_valid = protection_scores.dropna(subset=["protection_wape"])
complete_protection_origins = max(0, EVALUATION_WEEKS_N - base_accuracy_horizon + 1)
check(
    "Final-evaluation protection-period depth",
    (complete_protection_origins >= 5) and (not protection_valid.empty),
    f"{complete_protection_origins} complete {base_accuracy_horizon}-week origin(s) fit inside the "
    f"{EVALUATION_WEEKS_N}-week final evaluation; "
    f"protection-period scores are {'available' if not protection_valid.empty else 'not available'}. "
    "Fewer than 5 complete origins should be treated as a supporting horizon diagnostic rather than a primary accuracy result.",
    level="WARN",
)

if protection_valid.empty:
    protection_summary = pd.DataFrame(
        columns=[
            "weighted_protection_wape", "revenue_weighted_protection_wape",
            "median_sku_protection_wape", "revenue_weight_covered", "n_skus",
        ]
    )
    protection_seg_summary = pd.DataFrame()
    protection_revenue_seg_summary = pd.DataFrame()
else:
    protection_summary = (
        protection_valid
        .groupby("method")
        .apply(lambda g: pd.Series({
            "weighted_protection_wape": weighted_mean(g["protection_wape"], g["actual_units"]),
            "revenue_weighted_protection_wape": weighted_mean(g["protection_wape"], g["revenue_weight"]),
            "median_sku_protection_wape": g["protection_wape"].median(),
            "revenue_weight_covered": g.loc[g["protection_wape"].notna(), "revenue_weight"].sum(),
            "n_skus": len(g),
        }))
        .sort_values("revenue_weighted_protection_wape")
    )
    protection_seg_summary = (
        protection_valid
        .groupby(["segment", "method"])
        .apply(lambda g: weighted_mean(g["protection_wape"], g["actual_units"]))
        .unstack("method")
    )
    protection_seg_summary = protection_seg_summary.reindex(
        columns=[m for m in ALL_METHODS if m in protection_seg_summary.columns]
    )
    protection_revenue_seg_summary = (
        protection_valid
        .groupby(["segment", "method"])
        .apply(lambda g: weighted_mean(g["protection_wape"], g["revenue_weight"]))
        .unstack("method")
    )
    protection_revenue_seg_summary = protection_revenue_seg_summary.reindex(
        columns=[m for m in ALL_METHODS if m in protection_revenue_seg_summary.columns]
    )

# Controlled all-method comparison on the same SKU set. This is the main
# final-evaluation accuracy comparison; the winners are descriptive ex-post
# accuracy winners for the same evaluation weeks used by the inventory model.
accuracy_common, accuracy_common_meta = restrict_to_common_sku_sample(
    scores,
    ALL_METHODS,
    metric_col="wape",
    context="final evaluation all-method accuracy comparison",
)
accuracy_common_summary = (
    accuracy_common.dropna(subset=["wape"])
    .groupby("method")
    .apply(lambda g: pd.Series({
        "revenue_weighted_wape": weighted_mean(g["wape"], g["revenue_weight"]),
        "demand_weighted_wape": weighted_mean(g["wape"], g["actual_units"]),
        "median_sku_wape": g["wape"].median(),
        "median_mase": g["mase"].median(),
    }))
    .sort_values("revenue_weighted_wape")
)
best_simple_accuracy = accuracy_common_summary.loc[
    accuracy_common_summary.index.intersection(SIMPLE_METHODS), "revenue_weighted_wape"
].idxmin()
best_ml_accuracy = accuracy_common_summary.loc[
    accuracy_common_summary.index.intersection(ML_METHODS), "revenue_weighted_wape"
].idxmin()

print(
    f"Controlled final-evaluation sample: {accuracy_common_meta['common_n_skus']} SKUs "
    f"covering {accuracy_common_meta['common_revenue_coverage']:.1%} of pre-evaluation revenue"
)
print("Best simple one-step method:", best_simple_accuracy)
print("Best ML one-step method:", best_ml_accuracy)
print("\nOne-step accuracy (common sample):")
print(accuracy_common_summary[[
    "revenue_weighted_wape", "demand_weighted_wape", "median_sku_wape", "median_mase"
]].round(3).to_string())
print("\nRevenue-weighted one-step WAPE by demand segment:")
print(revenue_seg_summary.round(3).to_string())
print(f"\nProtection-period accuracy ({base_accuracy_horizon} weeks; {complete_protection_origins} complete evaluation origins):")
print(protection_summary[[
    "revenue_weighted_protection_wape", "median_sku_protection_wape", "revenue_weight_covered", "n_skus"
]].round(3).to_string())


## 14. ML predictive fit and bias

Provides a compact supporting check on the two ML models. R² and revenue-weighted R² describe pooled predictive fit, while revenue-weighted bias shows whether forecasts systematically over- or under-predict. The key comparison is between the one-week horizon and the case-specific protection horizon.

These measures support interpretation of the forecasting results; they do not determine the inventory winner. Optional intermediate-horizon output can be enabled from `CONFIG` when additional diagnostic detail is useful.


In [ ]:
section(14, "ML diagnostics: fit, bias, and supporting interpretability")
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

ML_HORIZONS = sorted(set(
    h for h in [1, 4, 8, 12, 16, base_accuracy_horizon]
    if h <= base_accuracy_horizon and h <= EVALUATION_WEEKS_N
))
_actual_lookup_diag = panel.set_index(["sku", "week"])["units"].to_dict()
_ml_diag_common_skus = set(accuracy_common["sku"].unique())
_rows = []

for h in ML_HORIZONS:
    sub = rolling_test_forecasts[
        rolling_test_forecasts["method"].isin(ML_METHODS)
        & (rolling_test_forecasts["horizon_step"] <= h)
        & rolling_test_forecasts["sku"].isin(_ml_diag_common_skus)
    ]
    for (sku, method_name, origin_week), g in sub.groupby(["sku", "method", "origin_week"]):
        g = g.sort_values("horizon_step").drop_duplicates("horizon_step")
        if len(g) < h or not np.array_equal(
            g.iloc[:h]["horizon_step"].to_numpy(), np.arange(1, h + 1)
        ):
            continue
        g = g.iloc[:h]
        actual = np.array(
            [_actual_lookup_diag.get((sku, wk), np.nan) for wk in g["target_week"]],
            dtype=float,
        )
        if np.isnan(actual).any():
            continue
        _rows.append({
            "horizon_weeks": h,
            "origin_week": origin_week,
            "sku": sku,
            "method": method_name,
            "actual_sum": float(actual.sum()),
            "forecast_sum": float(g["forecast"].sum()),
            "revenue_weight": float(TEST_REVENUE_WEIGHTS.get(sku, 0.0)),
        })

ml_horizon_detail = pd.DataFrame(_rows)
_summary = []
for (h, method_name), g in ml_horizon_detail.groupby(["horizon_weeks", "method"]):
    y = g["actual_sum"].to_numpy(float)
    p = g["forecast_sum"].to_numpy(float)
    w = g["revenue_weight"].to_numpy(float)
    denom = float(np.sum(w * y))
    _summary.append({
        "horizon_weeks": h,
        "method": method_name,
        "revenue_weighted_wape": np.sum(w * np.abs(p - y)) / denom if denom > 0 else np.nan,
        "revenue_weighted_bias_pct": 100 * np.sum(w * (p - y)) / denom if denom > 0 else np.nan,
        "r2": r2_score(y, p) if len(g) > 1 and np.var(y) > 0 else np.nan,
        "revenue_weighted_r2": (
            r2_score(y, p, sample_weight=w)
            if len(g) > 1 and np.var(y) > 0 and w.sum() > 0
            else np.nan
        ),
        "complete_origins": g["origin_week"].nunique(),
    })

ml_horizon_summary = pd.DataFrame(_summary).sort_values(["horizon_weeks", "method"])
ml_model_performance = (
    ml_horizon_summary[ml_horizon_summary["horizon_weeks"] == 1]
    .set_index("method")[["r2", "revenue_weighted_r2"]]
    .rename(columns={
        "r2": "r2_equal_sku_week",
        "revenue_weighted_r2": "r2_revenue_weighted",
    })
)

# Compact thesis-facing diagnostic: short-horizon fit versus the actual
# protection horizon. Bias is kept because systematic over/under-forecasting
# can change inventory outcomes even when WAPE is similar.
key_ml_horizon_summary = ml_horizon_summary[
    ml_horizon_summary["horizon_weeks"].isin([1, base_accuracy_horizon])
].copy()

print("ML predictive diagnostics: one week versus the protection horizon")
print(key_ml_horizon_summary[[
    "horizon_weeks",
    "method",
    "revenue_weighted_bias_pct",
    "r2",
    "revenue_weighted_r2",
    "complete_origins",
]].round(3).to_string(index=False))

if CONFIG.get("show_full_ml_horizon_table", False):
    print("\nSupporting intermediate-horizon diagnostic:")
    print(ml_horizon_summary.round(3).to_string(index=False))

if CONFIG.get("plot_ml_horizon_diagnostic", False):
    fig, ax = plt.subplots(figsize=(8, 5))
    for method_name, g in ml_horizon_summary.groupby("method"):
        g = g.sort_values("horizon_weeks")
        ax.plot(
            g["horizon_weeks"],
            g["revenue_weighted_r2"],
            marker="o",
            label=method_name,
        )
    ax.axhline(0, linewidth=0.8)
    ax.set_xlabel("Forecast horizon (weeks)")
    ax.set_ylabel("Revenue-weighted cumulative-demand R²")
    ax.set_title("ML predictive fit across the decision horizon (supporting diagnostic)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES / "fig_ml_r2_by_horizon.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("\nIntermediate-horizon R² figure skipped (enable CONFIG['plot_ml_horizon_diagnostic'] for appendix output).")


## 15. Leakage-safe safety-stock calibration

Constructs historical forecast-error samples for safety-stock estimation using origins that occur strictly before the final evaluation period and whose full protection horizon is also realized before evaluation begins. This prevents safety-stock parameters from using future information.

The same rolling forecasting logic is applied to these calibration origins, and protection-period forecast errors are retained for method-specific buffering.


In [ ]:
section(15, "Leakage-safe safety-stock calibration")
evaluation_start = EVALUATION_START
all_weeks_ts = [pd.Timestamp(w) for w in weeks]

# Safety-stock calibration origins must satisfy two conditions:
# 1) at least MIN_TRAIN_WEEKS are available before the origin; and
# 2) the entire maximum protection horizon is realized before final evaluation.
safe_calibration_weeks = []
for i, w in enumerate(all_weeks_ts):
    enough_history = i >= MIN_TRAIN_WEEKS
    full_horizon_before_evaluation = (
        w < evaluation_start
        and w + pd.Timedelta(weeks=MAX_PROTECTION_HORIZON - 1) < evaluation_start
    )
    if enough_history and full_horizon_before_evaluation:
        safe_calibration_weeks.append(w)

max_calibration_origins = int(CONFIG.get("calibration_max_origins", 12))
calibration_weeks = safe_calibration_weeks[-max_calibration_origins:]
if len(calibration_weeks) < 2:
    raise ValueError(
        f"Only {len(calibration_weeks)} leakage-safe calibration origin(s) are available. "
        "At least two are required to estimate method-specific empirical safety stock. "
        "Reduce the maximum lead-time grid, shorten the final evaluation, or provide more history."
    )

check(
    "Safety-stock calibration depth",
    len(calibration_weeks) >= 8,
    f"{len(calibration_weeks)} leakage-safe pre-evaluation origin(s) are available for empirical safety-stock calibration. "
    "Fewer than 8 origins makes high empirical quantiles particularly sensitive to individual historical errors.",
    level="WARN",
)

rolling_calibration_forecasts = build_rolling_forecast_cache(
    calibration_weeks,
    MAX_PROTECTION_HORIZON,
    include_ml=True,
)

print("Safe pre-evaluation calibration origins used for safety stock:", len(calibration_weeks))
print("Calibration window:", min(calibration_weeks).date(), "to", max(calibration_weeks).date())
print("Rolling calibration forecast rows:", len(rolling_calibration_forecasts))

# Historical protection-period accuracy is reported only as a calibration
# diagnostic. The main accuracy results come from the final evaluation period.
calibration_protection_scores = score_protection_period_rolling(
    rolling_calibration_forecasts,
    base_accuracy_horizon,
)
calibration_protection_scores["revenue_weight"] = (
    calibration_protection_scores["sku"].map(TEST_REVENUE_WEIGHTS).fillna(0.0)
)
calibration_protection_valid = calibration_protection_scores.dropna(subset=["protection_wape"])

check(
    "Calibration protection-period accuracy availability",
    not calibration_protection_valid.empty,
    f"Protection-period accuracy for horizon={base_accuracy_horizon} weeks is "
    f"{'available' if not calibration_protection_valid.empty else 'not computable'} on the leakage-safe calibration origins.",
    level="WARN",
)

if calibration_protection_valid.empty:
    calibration_protection_summary = pd.DataFrame(
        columns=[
            "weighted_protection_wape", "revenue_weighted_protection_wape",
            "median_sku_protection_wape", "revenue_weight_covered", "n_skus",
        ]
    )
else:
    calibration_protection_summary = (
        calibration_protection_valid
        .groupby("method")
        .apply(lambda g: pd.Series({
            "weighted_protection_wape": weighted_mean(g["protection_wape"], g["actual_units"]),
            "revenue_weighted_protection_wape": weighted_mean(g["protection_wape"], g["revenue_weight"]),
            "median_sku_protection_wape": g["protection_wape"].median(),
            "revenue_weight_covered": g.loc[g["protection_wape"].notna(), "revenue_weight"].sum(),
            "n_skus": len(g),
        }))
        .sort_values("revenue_weighted_protection_wape")
    )

print(
    f"\nProtection-period accuracy on PRE-EVALUATION calibration origins, "
    f"horizon = {base_accuracy_horizon} weeks "
    f"({min(calibration_weeks).date()} to {max(calibration_weeks).date()}, "
    f"{len(calibration_weeks)} origins):"
)
print(calibration_protection_summary.round(3).to_string())


## 16. Method-specific safety stock

Converts historical protection-period forecast errors into a safety-stock quantity for each forecasting method and SKU. The main specification uses empirical error quantiles at the configured service target, with progressively broader pooling when a SKU has insufficient calibration observations. A normal-approximation option is retained as a robustness specification.

Forecasting methods therefore receive buffers calibrated from their own historical error distributions while all other inventory-policy rules remain common.


In [ ]:
section(16, "Method-specific safety stock")
from scipy.stats import norm as normal_dist

SAFETY_CACHE = {}


def protection_horizon(params):
    return int(params["lead_time_weeks"]) + int(params["review_period_weeks"])


def safety_stock_from_errors(errors, service_level, method="empirical"):
    errors = np.asarray(errors, dtype=float)
    errors = errors[~np.isnan(errors)]
    if len(errors) == 0:
        # No calibration data at any pooling level (sku, segment, or method-wide).
        # Returning 0.0 here would be indistinguishable from "this method's
        # forecasts are so reliable it needs no buffer" -- a completely
        # different claim from "we have no idea." Return NaN so this gap is
        # visible and handled explicitly downstream, instead of silently
        # asserting zero risk.
        return np.nan
    if method == "empirical":
        return float(max(0.0, np.quantile(errors, service_level)))
    if method == "normal":
        mu = float(np.mean(errors))
        sigma = float(np.std(errors, ddof=1)) if len(errors) > 1 else 0.0
        return float(max(0.0, mu + normal_dist.ppf(service_level) * sigma))
    raise ValueError("method must be 'empirical' or 'normal'")


def calibration_errors_from_cache(forecast_cache, horizon):
    if forecast_cache is None or len(forecast_cache) == 0:
        return pd.DataFrame(columns=["sku", "method", "segment", "origin_week", "error"])
    actual_lookup = panel.set_index(["sku", "week"])["units"].to_dict()
    rows = []
    sub = forecast_cache[forecast_cache["horizon_step"] <= horizon].copy()
    for (sku, method_name, origin_week), g in sub.groupby(["sku", "method", "origin_week"]):
        g = g.sort_values("horizon_step")
        actuals = np.array([actual_lookup.get((sku, wk), np.nan) for wk in g["target_week"]], dtype=float)
        if len(actuals) < horizon or np.isnan(actuals).any():
            continue
        rows.append({
            "sku": sku,
            "method": method_name,
            "segment": sku_segment.get(sku, "unknown"),
            "origin_week": origin_week,
            "error": float(actuals[:horizon].sum() - g["forecast"].values[:horizon].sum()),
        })
    return pd.DataFrame(rows)


def rolling_origin_errors_simple(history, method_fn, horizon, min_train=MIN_TRAIN_WEEKS, step=4, max_origins=8):
    # Fallback only if no pre-evaluation calibration cache is available.
    history = np.asarray(history, dtype=float)
    origins = list(range(min_train, len(history) - horizon + 1, step))
    if len(origins) > max_origins:
        origins = origins[-max_origins:]
    errors = []
    for origin in origins:
        fc = np.clip(method_fn(pd.Series(history[:origin]), horizon), 0, None)
        actual_h = history[origin:origin + horizon].sum()
        errors.append(actual_h - fc.sum())
    return np.asarray(errors, dtype=float)


CALIBRATION_ERROR_POOL_CACHE = {}


def get_calibration_error_pools(horizon):
    """Pre-grouped calibration error pools, cached by horizon alone.

    These pools depend only on `horizon`, not on service_level or
    safety_stock_method -- but were being recomputed (a real groupby over
    the full calibration forecast cache) every time compute_safety_stock_table
    was called. Section 15 calls this at 4 different service levels for the
    SAME horizon, so this was redone 4x for no reason. Different quantiles
    are still applied fresh downstream; only the underlying error pools are
    reused across calls that share a horizon.
    """
    horizon = int(horizon)
    if horizon in CALIBRATION_ERROR_POOL_CACHE:
        return CALIBRATION_ERROR_POOL_CACHE[horizon]

    err_df = calibration_errors_from_cache(rolling_calibration_forecasts, horizon)

    if err_df.empty:
        fallback_rows = []
        for sku in SKUS:
            hist = train[train["sku"] == sku].sort_values("week")["units"].values.astype(float)
            for method_name, fn in SIMPLE_FNS.items():
                for e in rolling_origin_errors_simple(hist, fn, horizon):
                    fallback_rows.append({
                        "sku": sku,
                        "method": method_name,
                        "segment": sku_segment.get(sku, "unknown"),
                        "origin_week": pd.NaT,
                        "error": e,
                    })
        err_df = pd.DataFrame(fallback_rows)

    pools = {
        "sku_method": {k: g["error"].values for k, g in err_df.groupby(["sku", "method"])},
        "segment_method": {k: g["error"].values for k, g in err_df.groupby(["segment", "method"])},
        "method": {k: g["error"].values for k, g in err_df.groupby("method")},
    }
    CALIBRATION_ERROR_POOL_CACHE[horizon] = pools
    return pools


def compute_safety_stock_table(horizon, service_level, method="empirical"):
    key = (int(horizon), round(float(service_level), 5), method)
    if key in SAFETY_CACHE:
        return SAFETY_CACHE[key].copy()

    pools = get_calibration_error_pools(horizon)
    err_by_sku_method = pools["sku_method"]
    err_by_segment_method = pools["segment_method"]
    err_by_method = pools["method"]

    rows = []
    for sku in SKUS:
        segment = sku_segment.get(sku, "unknown")
        for method_name in ALL_METHODS:
            errs = err_by_sku_method.get((sku, method_name), np.array([]))
            source = "sku"
            if len(errs) < 2:
                errs = err_by_segment_method.get((segment, method_name), np.array([]))
                source = "segment"
            if len(errs) < 2:
                errs = err_by_method.get(method_name, np.array([]))
                source = "method"
            rows.append({
                "sku": sku,
                "method": method_name,
                "horizon": horizon,
                "service_level": service_level,
                "safety_stock_method": method,
                "safety_source": source if len(errs) >= 2 else "none",
                "n_error_origins": len(errs),
                "safety_stock": safety_stock_from_errors(errs, service_level, method),
            })
    out = pd.DataFrame(rows)
    SAFETY_CACHE[key] = out.copy()
    return out


base_params = CONFIG.copy()
base_horizon = protection_horizon(base_params)
base_safety = compute_safety_stock_table(
    base_horizon,
    base_params["service_level"],
    base_params["safety_stock_method"],
)

print("Base protection horizon:", base_horizon)
print("Safety method:", base_params["safety_stock_method"])
print("\nAverage safety stock by method:")
print(base_safety.groupby("method")[["safety_stock", "n_error_origins"]].mean().round(2).to_string())

SIMULATION_CACHE = {}


def run_operational_simulation_cached(params, safety_table, forecast_cache=None):
    """Cached wrapper around run_operational_simulation.

    The base scenario gets fully re-simulated 3 separate times as-is: once in
    Section 13, once in Section 15 when service_target equals the base
    service_level, once in Section 16 when (lead_time, batch) equals the base
    scenario. Cached by every parameter that affects the simulated
    trajectory. Deliberately includes stockout_cost_mult even though it
    doesn't affect the trajectory itself -- it IS baked into the raw
    total_cost/stockout_cost this function returns, so leaving it out of the
    key would risk a stale, wrong cost hit if any future call site ever uses
    a different value. Costs nothing today (it's constant everywhere), just
    removes that risk.
    """
    key = (
        int(params["lead_time_weeks"]),
        int(params["review_period_weeks"]),
        round(float(params["service_level"]), 6),
        round(float(params.get("order_batch_multiplier", 0.0)), 6),
        round(float(params["stockout_cost_mult"]), 6),
        params.get("safety_stock_method", "empirical"),
        round(float(params.get("terminal_holding_weeks", 0.0)), 6),
    )
    if key in SIMULATION_CACHE:
        return SIMULATION_CACHE[key].copy()
    if forecast_cache is None:
        result = run_operational_simulation(params, safety_table)
    else:
        result = run_operational_simulation(params, safety_table, forecast_cache)
    SIMULATION_CACHE[key] = result.copy()
    return result


## 17. Weekly inventory simulation

Translates each forecast into replenishment decisions through the same periodic-review, order-up-to policy. At each review, the target inventory position equals expected demand over the protection period plus method-specific safety stock. Orders arrive after the configured lead time and are rounded to the case-specific batch rule when applicable.

The simulation records achieved service, units short, average inventory, and normalized holding-plus-shortage cost. Unmet demand is treated as lost sales, and the same initialization and terminal-inventory rules are applied to every forecasting method.


In [ ]:
section(17, "Weekly inventory simulation")
def batch_qty_for_sku(sku, params):
    multiplier = float(params.get("order_batch_multiplier", 0.0))
    if multiplier <= 0:
        return 0.0
    return float(max(1.0, np.ceil(avg_weekly_demand.get(sku, 0.0) * multiplier)))


def apply_batch_constraint(raw_order_qty, sku, params):
    """Apply the configured batch-rounding rule.

    `order_batch_multiplier` is interpreted as a batch size measured in weeks
    of that SKU's average demand. A positive desired order is rounded UP to the
    next complete batch multiple. A value of 0 disables batching.
    """
    raw_order_qty = float(max(0.0, raw_order_qty))
    if raw_order_qty <= 0:
        return 0.0, 0.0, batch_qty_for_sku(sku, params)

    batch_qty = batch_qty_for_sku(sku, params)
    if batch_qty <= 0:
        return raw_order_qty, 0.0, batch_qty

    order_qty = float(np.ceil(raw_order_qty / batch_qty) * batch_qty)
    return order_qty, order_qty - raw_order_qty, batch_qty


_FORECAST_INDEX_CACHE = {}
_ACTUAL_SERIES_CACHE = {}


def build_actual_series_index(actual_table):
    """
    Pre-index realized demand by SKU.

    Avoids repeatedly filtering and sorting the full actual-demand
    DataFrame for every SKU, method, and simulation scenario.

    This is a performance optimization only:
    the weeks and demand values used by the simulation are unchanged.
    """
    key = id(actual_table)

    cached = _ACTUAL_SERIES_CACHE.get(key)

    if cached is not None:
        return cached

    idx = {}

    actual_sorted = actual_table.sort_values(
        ["sku", "week"]
    )

    for sku, g in actual_sorted.groupby(
        "sku",
        sort=False,
    ):
        idx[sku] = (
            list(g["week"]),
            g["units"].to_numpy(dtype=float),
        )

    _ACTUAL_SERIES_CACHE[key] = idx

    return idx

def build_forecast_index(forecast_cache):
    """Pre-index a forecast cache as {(sku, method, origin_week): sorted forecast array}.

    Called millions of times across the sweep, forecast_sum_from_origin used
    to re-scan the entire forecast cache (four boolean conditions) on every
    single call -- by far the dominant cost of the simulation/sweep blocks
    at this SKU count. Indexed by id() so a cache reused across every sweep
    scenario (e.g. rolling_test_forecasts) only gets indexed once. Verified:
    identical output to the original, including NaN and missing-data cases.
    """
    key = id(forecast_cache)
    idx = _FORECAST_INDEX_CACHE.get(key)
    if idx is not None:
        return idx
    idx = {}
    sub = forecast_cache.sort_values("horizon_step")
    for (sku, method_name, origin_week), g in sub.groupby(["sku", "method", "origin_week"], sort=False):
        idx[(sku, method_name, origin_week)] = g["forecast"].values.astype(float)
    _FORECAST_INDEX_CACHE[key] = idx
    return idx


def forecast_sum_from_origin(forecast_cache, sku, method_name, origin_week, horizon):
    idx = build_forecast_index(forecast_cache)
    vals = idx.get((sku, method_name, origin_week))
    if vals is None or len(vals) < horizon:
        return np.nan
    vals_h = vals[:horizon]
    if np.isnan(vals_h).any():
        return np.nan
    return float(vals_h.sum())


# Accumulates (sku, method) pairs skipped because no safety-stock calibration
# data existed at any pooling level. Reset and inspected after each simulation
# run so these gaps are visible rather than silently absorbed as zero risk.
SKIPPED_NO_SAFETY_STOCK = []


def simulate_policy_rolling(sku, method_name, forecast_cache, actual_table, params, safety_lookup):
    L = int(params["lead_time_weeks"])
    R = int(params["review_period_weeks"])
    if R < 1:
        raise ValueError("review_period_weeks must be at least 1.")
    H = protection_horizon(params)
    h_cost = float(params["holding_cost"])
    s_cost = h_cost * float(params["stockout_cost_mult"])
    safety_stock = float(safety_lookup.get((sku, method_name), np.nan))

    actual_index = build_actual_series_index(
    actual_table
    )

    actual_data = actual_index.get(sku)

    if actual_data is None:
        return None

    weeks_sku, actual = actual_data
    if len(actual) == 0 or actual.sum() == 0:
        return None
    if np.isnan(safety_stock):
        # No calibration data existed at any pooling level for this SKU/method
        # (see safety_stock_from_errors). Proceeding with an assumed buffer of
        # either 0 or some other stand-in would silently misrepresent this
        # combination's real stockout/service performance. Skip it entirely
        # and count it, rather than asserting a number we do not have.
        SKIPPED_NO_SAFETY_STOCK.append((sku, method_name))
        return None

    # Start at the method's target inventory position at the first review week.
    opening_forecast = forecast_sum_from_origin(forecast_cache, sku, method_name, weeks_sku[0], H)
    if np.isnan(opening_forecast):
        # No usable forecast even for the opening decision -- there is nothing
        # meaningful to simulate for this SKU/method, so skip it entirely
        # rather than starting from a fabricated zero-demand opening target.
        return None
    opening_target = opening_forecast + safety_stock
    on_hand = max(0.0, opening_target)
    pipeline = {}

    total_holding = 0.0
    total_stockout = 0.0
    units_short = 0.0
    inventory_sum = 0.0
    orders_placed = 0
    order_units = 0.0
    orders_arrived = 0
    arrived_order_units = 0.0
    batch_extra_units = 0.0
    weeks_skipped_no_forecast = 0
    review_opportunities = 0
    batch_size_used = batch_qty_for_sku(sku, params)

    for t, (week, demand) in enumerate(zip(weeks_sku, actual)):
        # Orders placed L weeks ago arrive at the start of the week.
        arriving_qty = float(pipeline.pop(t, 0.0))
        on_hand += arriving_qty
        if arriving_qty > 0:
            orders_arrived += 1
            arrived_order_units += arriving_qty

        # The target level (H = lead time + review period) already assumes an
        # order placed today has to last until the NEXT review opportunity --
        # but the simulation previously reviewed and ordered every single week
        # regardless of review_period_weeks, contradicting that assumption.
        # Only make a fresh ordering decision on actual review weeks.
        is_review_week = (t % R == 0)
        if is_review_week:
            review_opportunities += 1
            week_forecast = forecast_sum_from_origin(forecast_cache, sku, method_name, week, H)
            if np.isnan(week_forecast):
                # Missing forecast for this review week (e.g. a skipped ML
                # failure). Do NOT fabricate a 0-demand order-up-to target --
                # skip placing an order this week and count it, rather than
                # silently under-ordering and creating a phantom stockout.
                weeks_skipped_no_forecast += 1
            else:
                target_position = week_forecast + safety_stock
                inventory_position = on_hand + sum(pipeline.values())
                raw_order = max(0.0, target_position - inventory_position)
                order_qty, extra, batch_size_used = apply_batch_constraint(raw_order, sku, params)
                if order_qty > 0:
                    arrival_idx = t + L
                    pipeline[arrival_idx] = pipeline.get(arrival_idx, 0.0) + order_qty
                    orders_placed += 1
                    order_units += order_qty
                    batch_extra_units += extra

        # Demand is realized after the replenishment decision for the week.
        if demand <= on_hand:
            on_hand -= demand
        else:
            short = demand - on_hand
            units_short += short
            total_stockout += short * s_cost
            on_hand = 0.0

        total_holding += on_hand * h_cost
        inventory_sum += on_hand

    # An order placed near the end of the horizon may not have arrived yet.
    # Previously only on_hand was counted at the end, so a method could place
    # a large late order -- a real, committed inventory obligation -- without
    # it ever appearing in its terminal inventory measure.
    outstanding_order_units = float(sum(pipeline.values()))
    terminal_inventory_position = on_hand + outstanding_order_units
    terminal_holding_cost = terminal_inventory_position * h_cost * float(params.get("terminal_holding_weeks", 0.0))
    total_holding += terminal_holding_cost
    total_demand = float(actual.sum())

    return {
        "total_cost": total_holding + total_stockout,
        "holding_cost": total_holding,
        "stockout_cost": total_stockout,
        "terminal_holding_cost": terminal_holding_cost,
        "fill_rate": 1.0 - units_short / total_demand if total_demand > 0 else 1.0,
        "units_short": units_short,
        "avg_inventory": inventory_sum / len(actual),
        "ending_inventory": on_hand,
        "outstanding_order_units": outstanding_order_units,
        "terminal_inventory_position": terminal_inventory_position,
        "orders_placed": orders_placed,
        "order_units": order_units,
        "orders_arrived": orders_arrived,
        "arrived_order_units": arrived_order_units,
        "orders_outstanding_at_end": max(0, orders_placed - orders_arrived),
        "avg_order_qty": order_units / orders_placed if orders_placed else 0.0,
        "batch_size_used": batch_size_used,
        "batch_extra_units": batch_extra_units,
        "safety_stock": safety_stock,
        "weeks_skipped_no_forecast": weeks_skipped_no_forecast,
        "review_opportunities": review_opportunities,
        "total_weeks_simulated": len(actual),
    }


def run_operational_simulation(params, safety_table, forecast_cache=rolling_test_forecasts):
    SKIPPED_NO_SAFETY_STOCK.clear()
    safety_lookup = safety_table.set_index(["sku", "method"])["safety_stock"].to_dict()
    # Precompute once -- this only depends on sku, not method, but was being
    # re-filtered from the full `test` table on every (sku, method) pair.
    actual_units_by_sku = test.groupby("sku")["units"].sum().to_dict()
    rows = []
    methods_available = sorted(forecast_cache["method"].dropna().unique())
    for sku in SKUS:
        for method_name in methods_available:
            res = simulate_policy_rolling(sku, method_name, forecast_cache, test, params, safety_lookup)
            if res is None:
                continue
            actual_units = float(actual_units_by_sku.get(sku, 0.0))
            average_weekly_units = float(avg_weekly_demand.get(sku, 0.0))
            res.update({
                "sku": sku,
                "method": method_name,
                "segment": sku_segment.get(sku, "unknown"),
                "actual_units": actual_units,
                # Historical revenue share makes performance on economically
                # important SKUs count more in portfolio-level evaluation.
                "revenue_weight": float(TEST_REVENUE_WEIGHTS.get(sku, 0.0)),
                # Normalize physical cost/inventory within each SKU before
                # applying revenue weights, avoiding double-counting SKU volume.
                "cost_per_unit_demand": res["total_cost"] / actual_units if actual_units > 0 else np.nan,
                "holding_cost_per_unit_demand": res["holding_cost"] / actual_units if actual_units > 0 else np.nan,
                "stockout_cost_per_unit_demand": res["stockout_cost"] / actual_units if actual_units > 0 else np.nan,
                "avg_inventory_weeks": res["avg_inventory"] / average_weekly_units if average_weekly_units > 0 else np.nan,
                "lead_time": params["lead_time_weeks"],
                "stockout_mult": params["stockout_cost_mult"],
                "order_batch_multiplier": params.get("order_batch_multiplier", 0.0),
                "service_level": params["service_level"],
                "safety_stock_method": params.get("safety_stock_method", "empirical"),
            })
            rows.append(res)
    if SKIPPED_NO_SAFETY_STOCK:
        print(
            f"Note: {len(SKIPPED_NO_SAFETY_STOCK)} SKU/method combination(s) excluded from this "
            f"simulation run -- no safety-stock calibration data was available at any pooling "
            f"level: {SKIPPED_NO_SAFETY_STOCK[:10]}{' ...' if len(SKIPPED_NO_SAFETY_STOCK) > 10 else ''}"
        )
    return pd.DataFrame(rows)


sim = run_operational_simulation_cached(base_params, base_safety)
print("Simulation rows:", len(sim))

# With a finite evaluation window, only early orders at long lead times can arrive
# before the simulation closes. Keep this explicit because it limits interpretation of long-lead-time cases.
print("Replenishment observability by lead time:")
for _lt in sorted(set([int(CONFIG["lead_time_weeks"])] + [int(x) for x in CONFIG["lead_grid"]])):
    _observable_origins = max(0, EVALUATION_WEEKS_N - _lt)
    print(f"  LT={_lt}: {_observable_origins} of {EVALUATION_WEEKS_N} order-origin weeks can arrive inside the evaluation")

_base_observable = max(0, EVALUATION_WEEKS_N - int(CONFIG["lead_time_weeks"]))
check(
    "Base-scenario replenishment observation depth",
    _base_observable >= 5,
    f"At LT={CONFIG['lead_time_weeks']}, only {_base_observable} of {EVALUATION_WEEKS_N} order-origin weeks can arrive before the evaluation closes.",
    level="WARN",
)


## 18. Reference-scenario operational results — standardized recursive pipeline

Runs the common inventory simulation under the case's reference operating assumptions. Methods are compared on a common reliable SKU sample after excluding combinations with excessive missing forecast/review weeks. The section reports service, inventory, and normalized cost and identifies the best lower-complexity and standardized recursive-ML alternatives for the reference scenario.


In [ ]:
section(18, "Base-scenario operational results — standardized recursive pipeline")
SKIP_SHARE_THRESHOLD = 0.20

def filter_reliable_simulation_rows(df, threshold=SKIP_SHARE_THRESHOLD):
    out = df.copy()
    out["skip_share"] = out["weeks_skipped_no_forecast"] / out["review_opportunities"].replace(0, np.nan)
    return out[out["skip_share"] <= threshold].copy()

sim_reliable = filter_reliable_simulation_rows(sim)

operational_common, operational_common_meta = restrict_to_common_sku_sample(
    sim_reliable, ALL_METHODS, metric_col="cost_per_unit_demand",
    context="base scenario all-method operational comparison",
)

def summarize_operational(df):
    return (
        df.groupby("method")
        .apply(lambda g: pd.Series({
            "revenue_weighted_cost_per_unit": weighted_mean(g["cost_per_unit_demand"], g["revenue_weight"]),
            "revenue_weighted_fill_rate": weighted_mean(g["fill_rate"], g["revenue_weight"]),
            "revenue_weighted_inventory_weeks": weighted_mean(g["avg_inventory_weeks"], g["revenue_weight"]),
            "holding_cost_per_unit": weighted_mean(g["holding_cost_per_unit_demand"], g["revenue_weight"]),
            "shortage_units_per_unit": weighted_mean(g["units_short"] / g["actual_units"].replace(0, np.nan), g["revenue_weight"]),
        }))
        .sort_values("revenue_weighted_cost_per_unit")
    )

operational_common_summary = summarize_operational(operational_common)
best_simple_operational = operational_common_summary.loc[
    operational_common_summary.index.intersection(SIMPLE_METHODS), "revenue_weighted_cost_per_unit"
].idxmin()
best_recursive_ml_operational = operational_common_summary.loc[
    operational_common_summary.index.intersection(ML_METHODS), "revenue_weighted_cost_per_unit"
].idxmin()

accuracy_winner_scoreboard = operational_common_summary.reindex([best_simple_accuracy, best_ml_accuracy])
operational_winner_scoreboard = operational_common_summary.reindex([best_simple_operational, best_recursive_ml_operational])

print(
    f"Controlled base sample: {operational_common_meta['common_n_skus']} SKUs, "
    f"{operational_common_meta['common_revenue_coverage']:.1%} revenue coverage"
)
print("\nAll-method operational scoreboard:")
print(operational_common_summary[[
    "revenue_weighted_cost_per_unit", "revenue_weighted_fill_rate", "revenue_weighted_inventory_weeks"
]].round(3).to_string())
print(f"\nAccuracy winners: {best_simple_accuracy} vs {best_ml_accuracy}")
print(accuracy_winner_scoreboard[["revenue_weighted_cost_per_unit", "revenue_weighted_fill_rate", "revenue_weighted_inventory_weeks"]].round(3).to_string())
print(f"\nOperational family winners (recursive architecture): {best_simple_operational} vs {best_recursive_ml_operational}")
print(operational_winner_scoreboard[["revenue_weighted_cost_per_unit", "revenue_weighted_fill_rate", "revenue_weighted_inventory_weeks"]].round(3).to_string())

base_simple_cost = operational_winner_scoreboard.loc[best_simple_operational, "revenue_weighted_cost_per_unit"]
base_ml_cost = operational_winner_scoreboard.loc[best_recursive_ml_operational, "revenue_weighted_cost_per_unit"]
base_ml_advantage_pct = 100 * (base_simple_cost - base_ml_cost) / base_simple_cost
print(f"Base best-in-family recursive-ML cost advantage: {base_ml_advantage_pct:.2f}% (positive = ML cheaper)")

# Compact weighting sensitivity: shows whether conclusions differ between equal-SKU and economic weighting.
weighting_sensitivity = (
    operational_common.groupby("method")
    .apply(lambda g: pd.Series({
        "equal_sku_cost_per_unit": g["cost_per_unit_demand"].mean(),
        "revenue_weighted_cost_per_unit": weighted_mean(g["cost_per_unit_demand"], g["revenue_weight"]),
        "equal_sku_fill_rate": g["fill_rate"].mean(),
        "revenue_weighted_fill_rate": weighted_mean(g["fill_rate"], g["revenue_weight"]),
        "equal_sku_inventory_weeks": g["avg_inventory_weeks"].mean(),
        "revenue_weighted_inventory_weeks": weighted_mean(g["avg_inventory_weeks"], g["revenue_weight"]),
    }))
    .reindex(operational_common_summary.index)
)

if CONFIG.get("show_weighting_sensitivity_table", True):
    print("\nEqual-SKU versus revenue-weighted operational results (supporting robustness):")
    print(weighting_sensitivity.round(3).to_string())

# Exact lower-envelope break-even between the simple and ML families.
method_lines = operational_common_summary[["holding_cost_per_unit", "shortage_units_per_unit"]].reset_index()
method_lines["family"] = np.where(method_lines["method"].isin(SIMPLE_METHODS), "simple", "recursive_ml")

def _family_best(multiplier, family):
    sub = method_lines[method_lines["family"] == family].copy()
    sub["cost"] = sub["holding_cost_per_unit"] + multiplier * sub["shortage_units_per_unit"]
    row = sub.loc[sub["cost"].idxmin()]
    return row["method"], float(row["cost"])

candidate_points = {0.0}
for i in range(len(method_lines)):
    for j in range(i + 1, len(method_lines)):
        hi, qi = method_lines.loc[i, ["holding_cost_per_unit", "shortage_units_per_unit"]]
        hj, qj = method_lines.loc[j, ["holding_cost_per_unit", "shortage_units_per_unit"]]
        if np.isclose(qi, qj):
            continue
        m = (hj - hi) / (qi - qj)
        if np.isfinite(m) and m >= 0:
            candidate_points.add(float(m))

family_break_even_rows = []
for m in sorted(candidate_points):
    sm, sc = _family_best(m, "simple")
    mm, mc = _family_best(m, "recursive_ml")
    if np.isclose(sc, mc, rtol=1e-9, atol=1e-10):
        family_break_even_rows.append({
            "break_even_stockout_mult": m,
            "simple_method": sm,
            "recursive_ml_method": mm,
            "cost_per_unit": (sc + mc) / 2,
        })
if family_break_even_rows:
    family_break_even_points = (
        pd.DataFrame(family_break_even_rows)
        .sort_values("break_even_stockout_mult")
        .reset_index(drop=True)
    )
else:
    family_break_even_points = pd.DataFrame(columns=[
        "break_even_stockout_mult", "simple_method",
        "recursive_ml_method", "cost_per_unit"
    ])
recursive_ml_first_break_even = family_break_even_points.head(1).copy()
print("\nFirst best-simple vs best-recursive-ML stockout-cost break-even:")
if len(recursive_ml_first_break_even):
    print(recursive_ml_first_break_even.round(3).to_string(index=False))
    print(
        "Interpretation: the multiplier is measured against one unit-week of holding cost; "
        "it is a break-even condition, not an observed company cost parameter."
    )
else:
    print("No finite non-negative crossing.")

check(
    "Primary comparison coverage",
    operational_common_meta["common_revenue_coverage"] >= CONFIG["min_common_revenue_coverage"],
    f"{operational_common_meta['common_n_skus']} common SKUs covering {operational_common_meta['common_revenue_coverage']:.1%} of pre-evaluation revenue",
    level="FAIL",
)


## 19. Decision-horizon ML comparison: recursive versus direct-H

Tests whether the ML target itself should be aligned directly with the replenishment decision horizon. In addition to the standardized one-week ML models extended recursively, direct-H Random Forest and gradient boosting models are trained to predict cumulative demand over the case-specific protection horizon `H = lead time + review period`.

Because the direct target is horizon-specific, this comparison is limited to the reference operating scenario. The wider lead-time sensitivity analysis continues to use the standardized recursive architecture.


In [ ]:
section(19, "Decision-horizon ML comparison — recursive versus direct-H")

# Standardized ML is trained one week ahead and recursively extended.
# Direct-H models predict cumulative demand over this SME's base protection
# period in one shot. They are base-scenario-only because a direct target is
# horizon-specific.
DIRECT_H = int(base_horizon)
DIRECT_TARGET_COL = f"target_demand_next_{DIRECT_H}w"
DIRECT_METHOD_MAP = {m: f"{m}_direct{DIRECT_H}" for m in ML_METHODS}
DIRECT_METHODS = list(DIRECT_METHOD_MAP.values())


def make_direct_target_frame(feature_frame, horizon):
    out = feature_frame.sort_values(["sku", "week"]).copy()
    g = out.groupby("sku")["units"]
    target = sum((g.shift(-k) for k in range(int(horizon))))
    out[DIRECT_TARGET_COL] = target
    out["direct_target_end_week"] = out["week"] + pd.to_timedelta(int(horizon) - 1, unit="W")
    return out


direct_feat = make_direct_target_frame(feat, DIRECT_H)


def fit_direct_ml_models(origin_week):
    origin_week = pd.Timestamp(origin_week)
    training = direct_feat[
        (direct_feat["week"] < origin_week)
        & (direct_feat["direct_target_end_week"] < origin_week)
    ].dropna(subset=["lag_1", DIRECT_TARGET_COL])
    if training.empty:
        return {}
    X = training[FEATURE_COLS].fillna(0.0)
    y = training[DIRECT_TARGET_COL].astype(float)
    fitted = {}
    for base_name, make_model in ML_FACTORIES.items():
        try:
            fitted[DIRECT_METHOD_MAP[base_name]] = make_model().fit(X, y)
        except Exception as exc:
            print(f"Warning: {DIRECT_METHOD_MAP[base_name]} fit failed at {origin_week.date()}: {exc}")
    return fitted


def build_direct_forecasts(origin_weeks, phase_label):
    rows, cached_models = [], None
    for i, origin_week in enumerate(origin_weeks, start=1):
        origin_week = pd.Timestamp(origin_week)
        if cached_models is None or (i - 1) % REFIT_EVERY_N_ORIGINS == 0:
            cached_models = fit_direct_ml_models(origin_week)
        origin_rows = feat[
            (feat["week"] == origin_week) & feat["sku"].isin(SKUS)
        ].dropna(subset=["lag_1"])
        if origin_rows.empty or not cached_models:
            continue
        X = origin_rows[FEATURE_COLS].fillna(0.0)
        future_weeks = pd.date_range(origin_week, periods=DIRECT_H, freq="W-MON")
        for method_name, model in cached_models.items():
            totals = np.clip(model.predict(X), 0, None)
            for sku, total in zip(origin_rows["sku"], totals):
                component = float(total) / DIRECT_H
                rows.extend({
                    "origin_week": origin_week,
                    "target_week": target_week,
                    "horizon_step": step,
                    "sku": sku,
                    "method": method_name,
                    "forecast": component,
                } for step, target_week in enumerate(future_weeks, start=1))
    print(f"Direct-H {phase_label} forecast rows: {len(rows)}")
    return pd.DataFrame(rows)


direct_eval_forecasts = build_direct_forecasts(test_weeks, "evaluation")
direct_calibration_forecasts = build_direct_forecasts(calibration_weeks, "calibration")
print(f"Direct target: cumulative demand over {DIRECT_H} weeks")

# Compare the strongest simple base method, recursive ML, and direct-H ML.
ROBUSTNESS_RECURSIVE_METHODS = list(dict.fromkeys([best_simple_operational, "ml_gbm", "ml_rf"]))
ROBUSTNESS_METHODS = ROBUSTNESS_RECURSIVE_METHODS + DIRECT_METHODS


def _with_revenue_weights(df):
    out = df.copy()
    out["revenue_weight"] = out["sku"].map(TEST_REVENUE_WEIGHTS).fillna(0.0)
    return out


direct_eval_scores = _with_revenue_weights(score_protection_period_rolling(direct_eval_forecasts, DIRECT_H))
direct_calibration_scores = _with_revenue_weights(score_protection_period_rolling(direct_calibration_forecasts, DIRECT_H))

eval_scores = pd.concat([
    protection_scores[protection_scores["method"].isin(ROBUSTNESS_RECURSIVE_METHODS)],
    direct_eval_scores,
], ignore_index=True)
eval_common, _ = restrict_to_common_sku_sample(
    eval_scores, ROBUSTNESS_METHODS, metric_col="protection_wape",
    context="direct-H evaluation comparison",
)
eval_wape = eval_common.groupby("method").apply(
    lambda g: weighted_mean(g["protection_wape"], g["revenue_weight"])
).rename("eval_revenue_weighted_protection_wape")

cal_scores = pd.concat([
    calibration_protection_scores[
        calibration_protection_scores["method"].isin(ROBUSTNESS_RECURSIVE_METHODS)
    ],
    direct_calibration_scores,
], ignore_index=True)
cal_common, _ = restrict_to_common_sku_sample(
    cal_scores, ROBUSTNESS_METHODS, metric_col="protection_wape",
    context="direct-H calibration comparison",
)
cal_wape = cal_common.groupby("method").apply(
    lambda g: weighted_mean(g["protection_wape"], g["revenue_weight"])
).rename("calibration_revenue_weighted_protection_wape")


def cumulative_origin_r2(forecast_cache, methods, horizon, allowed_skus):
    actual_lookup = panel.set_index(["sku", "week"])["units"].to_dict()
    rows = []
    sub = forecast_cache[
        forecast_cache["method"].isin(methods) & (forecast_cache["horizon_step"] <= horizon)
    ]
    for (sku, method_name, origin_week), g in sub.groupby(["sku", "method", "origin_week"]):
        g = g.sort_values("horizon_step").drop_duplicates("horizon_step")
        if sku not in allowed_skus or len(g) < horizon:
            continue
        g = g.iloc[:horizon]
        actual = np.array([actual_lookup.get((sku, wk), np.nan) for wk in g["target_week"]], dtype=float)
        if np.isnan(actual).any():
            continue
        rows.append({
            "method": method_name,
            "actual": float(actual.sum()),
            "forecast": float(g["forecast"].sum()),
            "weight": float(TEST_REVENUE_WEIGHTS.get(sku, 0.0)),
        })
    detail = pd.DataFrame(rows)
    out = {}
    for method_name, g in detail.groupby("method"):
        y, p, w = g["actual"].to_numpy(float), g["forecast"].to_numpy(float), g["weight"].to_numpy(float)
        out[method_name] = r2_score(y, p, sample_weight=w) if len(g) > 1 and np.var(y) > 0 and w.sum() > 0 else np.nan
    return pd.Series(out, name="eval_revenue_weighted_protection_r2")

combined_eval_forecasts = pd.concat([
    rolling_test_forecasts[rolling_test_forecasts["method"].isin(ROBUSTNESS_RECURSIVE_METHODS)],
    direct_eval_forecasts,
], ignore_index=True)
eval_r2 = cumulative_origin_r2(
    combined_eval_forecasts, ROBUSTNESS_METHODS, DIRECT_H, set(eval_common["sku"].unique())
)

# Direct-H models receive their own leakage-safe method-specific safety stock.
direct_errors = calibration_errors_from_cache(direct_calibration_forecasts, DIRECT_H)

def direct_safety_table(error_df):
    sku_pool = {k: g["error"].values for k, g in error_df.groupby(["sku", "method"])}
    seg_pool = {k: g["error"].values for k, g in error_df.groupby(["segment", "method"])}
    method_pool = {k: g["error"].values for k, g in error_df.groupby("method")}
    rows = []
    for sku in SKUS:
        segment = sku_segment.get(sku, "unknown")
        for method_name in DIRECT_METHODS:
            errs = sku_pool.get((sku, method_name), np.array([]))
            source = "sku"
            if len(errs) < 2:
                errs, source = seg_pool.get((segment, method_name), np.array([])), "segment"
            if len(errs) < 2:
                errs, source = method_pool.get(method_name, np.array([])), "method"
            rows.append({
                "sku": sku,
                "method": method_name,
                "horizon": DIRECT_H,
                "service_level": base_params["service_level"],
                "safety_stock_method": base_params["safety_stock_method"],
                "safety_source": source if len(errs) >= 2 else "none",
                "n_error_origins": len(errs),
                "safety_stock": safety_stock_from_errors(
                    errs, base_params["service_level"], base_params["safety_stock_method"]
                ),
            })
    return pd.DataFrame(rows)


direct_safety = direct_safety_table(direct_errors)
direct_sim = filter_reliable_simulation_rows(
    run_operational_simulation(base_params, direct_safety, direct_eval_forecasts)
)
robustness_sim = pd.concat([
    sim_reliable[sim_reliable["method"].isin(ROBUSTNESS_RECURSIVE_METHODS)],
    direct_sim,
], ignore_index=True)
robustness_common, robustness_meta = restrict_to_common_sku_sample(
    robustness_sim, ROBUSTNESS_METHODS, metric_col="cost_per_unit_demand",
    context="direct-H base operational comparison",
)
robustness_operational_full = summarize_operational(robustness_common)
robustness_operational = robustness_operational_full[[
    "revenue_weighted_cost_per_unit", "revenue_weighted_fill_rate", "revenue_weighted_inventory_weeks"
]]

# This is the final base-case decision-horizon comparison. One-step metrics are
# intentionally not attached to Direct-H models because they predict a different target.
decision_horizon_summary = pd.DataFrame(index=ROBUSTNESS_METHODS)
decision_horizon_summary = (
    decision_horizon_summary
    .join(eval_wape)
    .join(eval_r2)
    .join(cal_wape)
    .join(robustness_operational)
    .sort_values("revenue_weighted_cost_per_unit")
)

print(
    f"\nFINAL BASE-CASE DECISION-HORIZON COMPARISON "
    f"({robustness_meta['common_n_skus']} common SKUs, {robustness_meta['common_revenue_coverage']:.1%} revenue coverage)"
)
print(decision_horizon_summary.round(3).to_string())

# Best ML architecture at the actual case-specific decision horizon: recursive or direct.
ALL_BASE_ML_ARCHITECTURES = ML_METHODS + DIRECT_METHODS
base_ml_candidates = decision_horizon_summary.loc[
    decision_horizon_summary.index.intersection(ALL_BASE_ML_ARCHITECTURES),
    "revenue_weighted_cost_per_unit",
].dropna()
best_ml_decision_horizon = base_ml_candidates.idxmin()
best_simple_decision_horizon = best_simple_operational

base_simple_decision_cost = float(decision_horizon_summary.loc[best_simple_decision_horizon, "revenue_weighted_cost_per_unit"])
base_ml_decision_cost = float(decision_horizon_summary.loc[best_ml_decision_horizon, "revenue_weighted_cost_per_unit"])
base_decision_ml_advantage_pct = 100 * (base_simple_decision_cost - base_ml_decision_cost) / base_simple_decision_cost

print(
    f"\nBest base-case simple method: {best_simple_decision_horizon} | "
    f"Best ML architecture at H={DIRECT_H}: {best_ml_decision_horizon}"
)
print(
    f"Decision-horizon ML cost advantage: {base_decision_ml_advantage_pct:.2f}% "
    f"(positive = ML cheaper)"
)

pair_rows = []
for base_name in ML_METHODS:
    direct_name = DIRECT_METHOD_MAP[base_name]
    if base_name not in robustness_operational.index or direct_name not in robustness_operational.index:
        continue
    rec, direct = robustness_operational.loc[base_name], robustness_operational.loc[direct_name]
    pair_rows.append({
        "algorithm": base_name,
        "cost_change_direct_vs_recursive_pct": 100 * (direct["revenue_weighted_cost_per_unit"] - rec["revenue_weighted_cost_per_unit"]) / rec["revenue_weighted_cost_per_unit"],
        "inventory_change_direct_vs_recursive_pct": 100 * (direct["revenue_weighted_inventory_weeks"] - rec["revenue_weighted_inventory_weeks"]) / rec["revenue_weighted_inventory_weeks"],
        "fill_rate_change_pp": 100 * (direct["revenue_weighted_fill_rate"] - rec["revenue_weighted_fill_rate"]),
    })
direct_vs_recursive = pd.DataFrame(pair_rows)
print("\nDirect-H versus recursive ML (same algorithm):")
print(direct_vs_recursive.round(2).to_string(index=False))

check(
    "Decision-horizon comparison coverage",
    robustness_meta.get("common_revenue_coverage", 0.0) >= MIN_COMMON_REVENUE_COVERAGE,
    f"{robustness_meta.get('common_n_skus', 0)} common SKUs covering {robustness_meta.get('common_revenue_coverage', 0.0):.1%} of pre-evaluation revenue",
    level="FAIL",
)


## 20. Inventory-service frontier — standardized recursive methods

Repeats the operational simulation across the configured safety-stock service targets. For each method, the resulting points show the trade-off between achieved fill rate and inventory held, allowing methods to be compared on service and stock requirements rather than on a single cost assumption alone.


In [ ]:
section(20, "Inventory-service frontier")
def run_service_frontier(base_params):
    rows = []
    for service in CONFIG["service_grid"]:
        params = dict(base_params)
        params["service_level"] = service
        horizon = protection_horizon(params)
        safety_table = compute_safety_stock_table(horizon, service, params["safety_stock_method"])
        sub = run_operational_simulation_cached(params, safety_table)
        sub = filter_reliable_simulation_rows(sub)
        # Restrict every service-target comparison to the same valid SKU set
        # across all methods so the inventory-service frontier is directly comparable.
        sub, _meta = restrict_to_common_sku_sample(
            sub, ALL_METHODS, metric_col="avg_inventory_weeks",
            context=f"inventory-service frontier target={service}",
        )
        sub["service_target"] = service
        rows.append(sub)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


frontier = run_service_frontier(base_params)
frontier_physical = (
    frontier.groupby(["method", "service_target"], as_index=False)
    .agg(
        avg_sku_fill_rate=("fill_rate", "mean"),
        total_actual_units=("actual_units", "sum"),
        total_units_short=("units_short", "sum"),
        avg_inventory=("avg_inventory", "mean"),
        total_cost=("total_cost", "sum"),
    )
)
frontier_physical["demand_weighted_fill_rate"] = (
    1.0 - frontier_physical["total_units_short"] / frontier_physical["total_actual_units"].replace(0, np.nan)
)

frontier_revenue = (
    frontier.groupby(["method", "service_target"])
    .apply(lambda g: pd.Series({
        "revenue_weighted_cost_per_unit": weighted_mean(g["cost_per_unit_demand"], g["revenue_weight"]),
        "revenue_weighted_fill_rate": weighted_mean(g["fill_rate"], g["revenue_weight"]),
        "revenue_weighted_inventory_weeks": weighted_mean(g["avg_inventory_weeks"], g["revenue_weight"]),
        "revenue_weight_covered": g.loc[g["cost_per_unit_demand"].notna(), "revenue_weight"].sum(),
    }))
    .reset_index()
)
frontier_summary = frontier_physical.merge(frontier_revenue, on=["method", "service_target"], how="left")
frontier_summary["achieved_fill_rate"] = frontier_summary["revenue_weighted_fill_rate"]
frontier_summary["frontier_inventory"] = frontier_summary["revenue_weighted_inventory_weeks"]
print("Reference service-target frontier points:")
print(frontier_summary[np.isclose(frontier_summary["service_target"], base_params["service_level"])][["method", "revenue_weighted_fill_rate", "revenue_weighted_inventory_weeks"]].round(3).to_string(index=False))


## 21. Operating-condition map — standardized recursive methods

Tests whether the operational ranking is robust to different operating conditions. The scenario sweep varies lead time, relative shortage cost, and order-batch constraints while keeping the forecasting architecture and inventory policy consistent.

For each scenario, the lowest-cost lower-complexity method is compared with the lowest-cost recursive ML method on the same valid SKU sample. Positive ML advantage means the ML family produces lower normalized inventory-related cost.


In [ ]:
section(21, "Operating-condition map — recursive ML, shortage cost, and order constraints")

def run_sweep(base_params):
    base_lead = base_params["lead_time_weeks"]
    base_batch = base_params["order_batch_multiplier"]
    combos = {(L, base_batch) for L in CONFIG["lead_grid"]} | {(base_lead, bm) for bm in CONFIG["batch_grid"]}
    rows = []
    for lead_time, batch_mult in combos:
        params = dict(base_params)
        params["lead_time_weeks"] = lead_time
        params["order_batch_multiplier"] = batch_mult
        safety_table = compute_safety_stock_table(
            protection_horizon(params), params["service_level"], params["safety_stock_method"]
        )
        base_sim = filter_reliable_simulation_rows(run_operational_simulation_cached(params, safety_table))
        for stockout_mult in CONFIG["stockout_grid"]:
            rep = base_sim.copy()
            rep["stockout_mult"] = stockout_mult
            rep["stockout_cost"] = rep["units_short"] * params["holding_cost"] * stockout_mult
            rep["total_cost"] = rep["holding_cost"] + rep["stockout_cost"]
            rep["cost_per_unit_demand"] = rep["total_cost"] / rep["actual_units"].replace(0, np.nan)
            rows.append(rep)
    return pd.concat(rows, ignore_index=True)


def best_family_by_scenario(df):
    rows = []
    for (lead_time, stockout_mult, batch_mult), g in df.groupby(["lead_time", "stockout_mult", "order_batch_multiplier"]):
        common, meta = restrict_to_common_sku_sample(
            g, ALL_METHODS, metric_col="cost_per_unit_demand",
            context=f"conditions map LT={lead_time}, cost={stockout_mult}, order_constraint={batch_mult}",
        )
        costs = common.groupby("method").apply(
            lambda x: weighted_mean(x["cost_per_unit_demand"], x["revenue_weight"])
        )
        simple = costs[costs.index.isin(SIMPLE_METHODS)]
        ml = costs[costs.index.isin(ML_METHODS)]
        if simple.empty or ml.empty:
            continue
        simple_method, ml_method = simple.idxmin(), ml.idxmin()
        simple_cost, ml_cost = float(simple.min()), float(ml.min())
        rows.append({
            "lead_time": lead_time,
            "stockout_mult": stockout_mult,
            "order_batch_multiplier": batch_mult,
            "simple_method": simple_method,
            "ml_method": ml_method,
            "simple_cost": simple_cost,
            "ml_cost": ml_cost,
            "ml_advantage_pct": 100 * (simple_cost - ml_cost) / simple_cost,
            "common_n_skus": meta["common_n_skus"],
            "common_revenue_coverage": meta["common_revenue_coverage"],
        })
    return pd.DataFrame(rows)

sweep = run_sweep(base_params)
conditions_best_family_long = best_family_by_scenario(sweep)
BASE_BATCH_FOR_MAP = float(base_params["order_batch_multiplier"])
map_long = conditions_best_family_long[np.isclose(conditions_best_family_long["order_batch_multiplier"], BASE_BATCH_FOR_MAP)]
advantage_best_family = map_long.pivot(index="lead_time", columns="stockout_mult", values="ml_advantage_pct")

print("Best recursive-ML vs best simple cost advantage (%) — positive = ML cheaper:")
print(advantage_best_family.round(1).to_string())
print("\nWinning recursive methods at the reference shortage-cost ratio:")
print(map_long[map_long["stockout_mult"] == base_params["stockout_cost_mult"]][
    ["lead_time", "simple_method", "ml_method", "ml_advantage_pct"]
].sort_values("lead_time").round(2).to_string(index=False))

order_constraint_sensitivity = conditions_best_family_long[
    (conditions_best_family_long["lead_time"] == base_params["lead_time_weeks"])
    & (conditions_best_family_long["stockout_mult"] == base_params["stockout_cost_mult"])
].copy()
order_constraint_sensitivity["order_batch_multiplier"] = order_constraint_sensitivity["order_batch_multiplier"].round(3)
order_constraint_sensitivity = order_constraint_sensitivity.drop_duplicates(subset=["order_batch_multiplier"]).sort_values("order_batch_multiplier")
print("\nOrder-constraint sensitivity at the reference lead time and shortage-cost ratio:")
print(order_constraint_sensitivity[["order_batch_multiplier", "simple_method", "ml_method", "ml_advantage_pct"]].round(2).to_string(index=False))

## 22. Summary figures

Creates the main visual outputs used to communicate the case results: the inventory-service frontier, the reference-scenario recursive-versus-direct ML comparison, and the operating-condition map showing where ML is more or less favorable than the best lower-complexity alternative.


In [ ]:
section(22, "Core thesis figures")
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# 1) Inventory-service frontier for standardized recursive methods.
fig, ax = plt.subplots(figsize=(8, 6))
for method_name, g in frontier_summary.groupby("method"):
    g = g.sort_values("service_target")
    ax.plot(
        g["revenue_weighted_inventory_weeks"],
        g["revenue_weighted_fill_rate"],
        marker="o",
        label=method_name,
    )
ax.set_xlabel("Revenue-weighted inventory (weeks of demand)")
ax.set_ylabel("Revenue-weighted achieved fill rate")
ax.set_title(
    "Inventory-service frontier by safety-stock calibration percentile\n"
    "Points represent the configured empirical protection-period error-coverage targets"
)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES / "fig_inventory_service_frontier.png", dpi=300, bbox_inches="tight")
plt.show()

# 2) Final base-case decision-horizon comparison, including direct-H ML.
plot_methods = [m for m in ROBUSTNESS_METHODS if m in decision_horizon_summary.index]
plot_df = decision_horizon_summary.loc[plot_methods].copy()
DISPLAY_NAMES = {
    best_simple_operational: best_simple_operational.upper(),
    "ml_gbm": "GBM recursive",
    "ml_rf": "RF recursive",
    DIRECT_METHOD_MAP.get("ml_gbm", ""): f"GBM direct-{DIRECT_H}",
    DIRECT_METHOD_MAP.get("ml_rf", ""): f"RF direct-{DIRECT_H}",
}

fig, ax = plt.subplots(figsize=(8, 6))
for method_name, row in plot_df.iterrows():
    x = row["revenue_weighted_inventory_weeks"]
    y = row["revenue_weighted_fill_rate"]
    ax.scatter(x, y, s=90)
    ax.annotate(
        DISPLAY_NAMES.get(method_name, method_name),
        (x, y),
        xytext=(6, 5),
        textcoords="offset points",
        fontsize=9,
    )
ax.set_xlabel("Revenue-weighted inventory (weeks of demand)")
ax.set_ylabel("Revenue-weighted fill rate")
ax.set_title(f"Base decision-horizon comparison (H={DIRECT_H} weeks)")
plt.tight_layout()
plt.savefig(FIGURES / "fig_direct_vs_recursive_operational_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

# 3) Best recursive ML vs best simple conditions map.
# Negative = simple cheaper = RED
# Positive = ML cheaper = BLUE
# Zero = neutral / white

fig, ax = plt.subplots(figsize=(8, 5))

data = advantage_best_family.sort_index().sort_index(axis=1)

vmin = float(np.nanmin(data.to_numpy()))
vmax = float(np.nanmax(data.to_numpy()))

# Safety in case all values are on one side of zero
if vmin >= 0:
    vmin = -1e-6
if vmax <= 0:
    vmax = 1e-6

norm = TwoSlopeNorm(
    vmin=vmin,
    vcenter=0.0,
    vmax=vmax
)

im = ax.imshow(
    data.to_numpy(),
    aspect="auto",
    norm=norm,
    cmap="RdBu",   # RED negative, BLUE positive
)

ax.set_xticks(
    range(len(data.columns)),
    [str(x) for x in data.columns]
)

ax.set_yticks(
    range(len(data.index)),
    [str(x) for x in data.index]
)

ax.set_xlabel("Stockout cost multiplier")
ax.set_ylabel("Lead time (weeks)")

ax.set_title(
    "Best recursive ML vs best simple: normalized-cost advantage"
)

# Percentage values inside cells
for i in range(len(data.index)):
    for j in range(len(data.columns)):
        value = data.iloc[i, j]
        if np.isfinite(value):
            ax.text(
                j,
                i,
                f"{value:.1f}%",
                ha="center",
                va="center",
                color="black"
            )

# Colorbar
cbar = fig.colorbar(im, ax=ax)

# Explicit percentage ticks including actual minimum and maximum
cbar_ticks = [
    vmin,
    -20,
    -15,
    -10,
    -5,
    0,
    5,
    10,
    15,
    20,
    vmax
]

# Keep only values inside the current case range
cbar_ticks = sorted(set(
    x for x in cbar_ticks
    if vmin <= x <= vmax
))

cbar.set_ticks(cbar_ticks)

cbar.set_ticklabels([
    f"{x:.1f}%" if np.isclose(x, vmin) or np.isclose(x, vmax)
    else f"{x:.0f}%"
    for x in cbar_ticks
])

cbar.set_label(
    "Recursive-ML cost advantage (positive = ML cheaper)",
    rotation=90,
    labelpad=20
)

plt.tight_layout()

plt.savefig(
    FIGURES / "fig_ml_advantage_best_in_family.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 23. Does forecast accuracy translate into operational value?

Links forecast improvement to simulated inventory value at SKU level. The selected simple and ML methods are compared on one-week WAPE and on normalized operational cost under the reference scenario. Correlations and quadrant counts show whether SKUs with better ML accuracy are also the SKUs for which ML reduces simulated cost.

This is a descriptive relationship, not a causal estimate. An optional shortage-cost sensitivity check examines whether the association changes materially under alternative cost assumptions.


In [ ]:
section(23, "Accuracy versus operational value")

# Use the one-step accuracy winners and the base operating scenario only.
acc_simple = accuracy_common[accuracy_common["method"] == best_simple_accuracy].set_index("sku")["wape"]
acc_ml = accuracy_common[accuracy_common["method"] == best_ml_accuracy].set_index("sku")["wape"]
accuracy_gain = (100 * (acc_simple - acc_ml) / acc_simple.replace(0, np.nan)).rename("accuracy_improvement_pct")

pair_sim, _ = restrict_to_common_sku_sample(
    sim_reliable[sim_reliable["method"].isin([best_simple_accuracy, best_ml_accuracy])],
    [best_simple_accuracy, best_ml_accuracy], metric_col="cost_per_unit_demand",
    context="accuracy-value base comparison",
)
simple_cost = pair_sim[pair_sim["method"] == best_simple_accuracy].set_index("sku")["cost_per_unit_demand"]
ml_cost = pair_sim[pair_sim["method"] == best_ml_accuracy].set_index("sku")["cost_per_unit_demand"]
symmetric_cost_advantage = (
    200 * (simple_cost - ml_cost) / (simple_cost + ml_cost).replace(0, np.nan)
).rename("symmetric_cost_advantage_pct")

div = pd.concat([accuracy_gain, symmetric_cost_advantage], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
div["segment"] = div.index.map(sku_segment)
div["revenue_weight"] = div.index.map(TEST_REVENUE_WEIGHTS).fillna(0.0)

pearson = div["accuracy_improvement_pct"].corr(div["symmetric_cost_advantage_pct"], method="pearson")
spearman = div["accuracy_improvement_pct"].corr(div["symmetric_cost_advantage_pct"], method="spearman")
weighted_pearson = weighted_corr(div["accuracy_improvement_pct"], div["symmetric_cost_advantage_pct"], div["revenue_weight"])
weighted_spearman = weighted_corr(
    div["accuracy_improvement_pct"].rank(),
    div["symmetric_cost_advantage_pct"].rank(),
    div["revenue_weight"],
)
print(
    f"Pearson={pearson:.3f} | Spearman={spearman:.3f} | "
    f"revenue-weighted Pearson={weighted_pearson:.3f} | "
    f"revenue-weighted Spearman={weighted_spearman:.3f}"
)

conditions = [
    (div["accuracy_improvement_pct"] >= 0) & (div["symmetric_cost_advantage_pct"] >= 0),
    (div["accuracy_improvement_pct"] >= 0) & (div["symmetric_cost_advantage_pct"] < 0),
    (div["accuracy_improvement_pct"] < 0) & (div["symmetric_cost_advantage_pct"] >= 0),
]
labels = [
    "ML better accuracy + lower cost",
    "ML better accuracy + higher cost",
    "ML worse accuracy + lower cost",
]
div["quadrant"] = np.select(conditions, labels, default="ML worse accuracy + higher cost")
quadrant_summary = div.groupby("quadrant").agg(
    n_skus=("quadrant", "size"),
    revenue_share=("revenue_weight", "sum"),
)
quadrant_summary["sku_share"] = quadrant_summary["n_skus"] / len(div)
print("\nAccuracy-value quadrants:")
print(quadrant_summary[["n_skus", "sku_share", "revenue_share"]].round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 6))
for segment, g in div.groupby("segment"):
    sizes = np.clip(25 + 2500 * g["revenue_weight"].to_numpy(), 25, 250)
    ax.scatter(
        g["accuracy_improvement_pct"],
        g["symmetric_cost_advantage_pct"],
        s=sizes,
        alpha=0.7,
        label=segment,
    )

# Historical-revenue-weighted linear trend. It is descriptive association,
# not a causal estimate.
trend = div[div["revenue_weight"] > 0].copy()
if len(trend) >= 2:
    x = trend["accuracy_improvement_pct"].to_numpy(float)
    y = trend["symmetric_cost_advantage_pct"].to_numpy(float)
    w = trend["revenue_weight"].to_numpy(float)
    coef = np.polyfit(x, y, deg=1, w=np.sqrt(w))
    x_line = np.linspace(x.min(), x.max(), 200)
    ax.plot(
        x_line,
        coef[0] * x_line + coef[1],
        linestyle="--",
        linewidth=2,
        label="Revenue-weighted trend",
    )

ax.axhline(0, linewidth=0.8)
ax.axvline(0, linewidth=0.8)
def _pretty_method_name(name):
    labels = {
        "naive": "Naive",
        "seasonal_naive": "Seasonal naive",
        "moving_avg": "Moving average",
        "ets": "ETS",
        "tsb": "TSB",
        "ml_gbm": "Gradient Boosting",
        "ml_rf": "Random Forest",
    }
    return labels.get(name, name)

simple_label = _pretty_method_name(best_simple_accuracy)
ml_label = _pretty_method_name(best_ml_accuracy)
ax.set_xlabel(f"{ml_label} one-step accuracy improvement vs {simple_label} (%)")
ax.set_ylabel(
    f"{ml_label} operational cost advantage vs {simple_label} (%)\n"
    f"positive = {ml_label} cheaper"
)
ax.set_title(
    "One-step accuracy gain vs SKU-level operational cost advantage\n"
    f"{simple_label} vs {ml_label} | "
    f"Revenue-weighted Pearson = {weighted_pearson:.3f} | "
    f"Spearman = {weighted_spearman:.3f}"
)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES / "fig_accuracy_vs_operational_value.png", dpi=300, bbox_inches="tight")
plt.show()

# Optional supporting sensitivity: does the accuracy/value association remain
# similar when the assumed stockout-cost ratio changes? The main thesis figure
# remains the reference-scenario scatter above.
accuracy_cost_correlation_sensitivity = pd.DataFrame()
if CONFIG.get("show_accuracy_cost_correlation_sensitivity", False):
    sensitivity_rows = []
    for stockout_mult in CONFIG["stockout_grid"]:
        scenario = sweep[
            (sweep["lead_time"] == base_params["lead_time_weeks"])
            & (sweep["stockout_mult"] == stockout_mult)
            & np.isclose(sweep["order_batch_multiplier"], base_params["order_batch_multiplier"])
            & sweep["method"].isin([best_simple_accuracy, best_ml_accuracy])
        ].copy()

        scenario_common, _ = restrict_to_common_sku_sample(
            scenario,
            [best_simple_accuracy, best_ml_accuracy],
            metric_col="cost_per_unit_demand",
            context=f"accuracy-value sensitivity cost={stockout_mult}",
        )
        sc = scenario_common[
            scenario_common["method"] == best_simple_accuracy
        ].set_index("sku")["cost_per_unit_demand"]
        mc = scenario_common[
            scenario_common["method"] == best_ml_accuracy
        ].set_index("sku")["cost_per_unit_demand"]
        cost_gain = (200 * (sc - mc) / (sc + mc).replace(0, np.nan)).rename("cost_gain")

        tmp = pd.concat([accuracy_gain, cost_gain], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
        tmp["revenue_weight"] = tmp.index.map(TEST_REVENUE_WEIGHTS).fillna(0.0)
        sensitivity_rows.append({
            "stockout_cost_mult": stockout_mult,
            "equal_sku_pearson": tmp["accuracy_improvement_pct"].corr(tmp["cost_gain"]),
            "revenue_weighted_pearson": weighted_corr(
                tmp["accuracy_improvement_pct"], tmp["cost_gain"], tmp["revenue_weight"]
            ),
        })

    accuracy_cost_correlation_sensitivity = pd.DataFrame(sensitivity_rows)
    print("\nSupporting accuracy/value correlation sensitivity:")
    print(accuracy_cost_correlation_sensitivity.round(3).to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(
        accuracy_cost_correlation_sensitivity["stockout_cost_mult"],
        accuracy_cost_correlation_sensitivity["equal_sku_pearson"],
        marker="o",
        label="Equal-SKU correlation",
    )
    ax.plot(
        accuracy_cost_correlation_sensitivity["stockout_cost_mult"],
        accuracy_cost_correlation_sensitivity["revenue_weighted_pearson"],
        marker="o",
        label="Historical-revenue-weighted correlation",
    )
    ax.axhline(0, linewidth=0.8)
    ax.set_xlabel("Assumed stockout cost (x holding cost)")
    ax.set_ylabel("Correlation: accuracy gain vs. cost advantage")
    ax.set_title("Accuracy/value correlation sensitivity (supporting)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(
        FIGURES / "fig_accuracy_vs_cost_correlation_by_ratio.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

## 24. Run health check

Collects the checks raised throughout the notebook into one final audit summary. Failures indicate that the run should not be interpreted until the underlying issue is resolved; warnings identify limitations or thin evidence that should be considered when reporting the case. The section also reports total runtime and the location of the saved logs.


In [ ]:
section(24, "Run health check")
print("=" * 70)
print("RUN HEALTH CHECK SUMMARY")
print("=" * 70)

n_fail = sum(1 for d in RUN_DIAGNOSTICS if d["status"] == "FAIL")
n_warn = sum(1 for d in RUN_DIAGNOSTICS if d["status"] == "WARN")
n_pass = sum(1 for d in RUN_DIAGNOSTICS if d["status"] == "PASS")

for d in RUN_DIAGNOSTICS:
    print(f"[{d['status']:4}] {d['check']}: {d['message']}")

print("-" * 70)
print(f"{n_pass} passed, {n_warn} warning(s), {n_fail} failure(s)")
notebook_elapsed_seconds = time.time() - NOTEBOOK_START_TIME
minutes = int(notebook_elapsed_seconds // 60)
seconds = notebook_elapsed_seconds % 60
print(
    f"Total notebook runtime: {minutes} min {seconds:.1f} sec "
    f"({notebook_elapsed_seconds / 60:.2f} minutes)"
)

if n_fail:
    print("\nAt least one check FAILED. Results below this point should not be trusted "
          "until the failing check above is understood and resolved.")
elif n_warn:
    print("\nNo failures, but one or more checks raised a warning above -- "
          "worth a quick look before treating the results as final.")
else:
    print("\nAll checks passed.")
print("=" * 70)

_section_logger.close()
print(f"\nAll per-section logs saved under: {LOGS}")

## Interpretation guide

The same analytical sequence is intended to be applied to each SME case, with only the input data and case-specific settings changed. The primary evidence is the portfolio structure, out-of-sample forecast performance, protection-horizon performance, reference-scenario inventory outcomes, recursive-versus-direct ML comparison, inventory-service frontier, operating-condition sensitivity, and SKU-level accuracy/value relationship.

The notebook is a pre-adoption simulation framework: it compares forecasting alternatives under controlled assumptions. It does not reconstruct the firm's historical inventory decisions or estimate the full financial return on implementing ML.
